<div align="center">

# 🫀 CELM-Cardio v8 — Revisi Sesuai Komentar Reviewer (IJIES 20266157)
## Ensemble EfficientNetB4 · DenseNet121 · ResNet50V2 + FPPR + XGBoost Stacking
### Cardiomegaly Classification · NIH ChestX-ray14 · Protokol Evaluasi Bebas Kebocoran

---

| Item | Detail |
|------|--------|
| **Dataset** | NIH ChestX-ray14 resized 224×224 (Kaggle · khanfashee) |
| **Kelas** | Cardiomegaly (label tunggal) · Non_Cardiomegaly (No Finding) |
| **Split** | **Per pasien**, 4 partisi terpisah: train 60% · val 10% · meta 15% · test 15% |
| **Backbone** | EfficientNetB4 · DenseNet121 · ResNet50V2 (ImageNet), Focal Loss, Cosine Decay |
| **Checkpoint** | Val-accuracy terbaik gabungan Fase 1 + Fase 2 (hanya partisi **val**) |
| **Level 1** | Rata-rata biasa (Mean) & FPPR — bobot dari akurasi **val** |
| **Level 2** | XGBoost dilatih **hanya** di partisi **meta** (tidak dilihat backbone), nested CV per pasien |
| **Pemilihan model** | Di partisi **meta** (AUC out-of-fold) — **test tidak dipakai untuk memilih** |
| **Test** | Dipakai **satu kali** untuk evaluasi akhir semua konfigurasi |
| **Statistik** | Bootstrap 95% CI per pasien · DeLong (AUC) · McNemar (accuracy) · koreksi Holm |
| **Kalibrasi** | Brier score · ECE · reliability diagram (AUC = diskriminasi, bukan kalibrasi) |
| **Uji eksternal** | CheXpert (Stanford) — A: setara protokol NIH · B: realistis (komorbid & kelainan lain); model & threshold terkunci |
| **CTR** | Diukur otomatis (segmentasi jantung & paru) → baseline klinis CTR>0.50 & analisis galat FN/FP |

</div>

## 📝 Peta Revisi → Komentar Reviewer

| Komentar | Isu | Perbaikan di notebook ini |
|---|---|---|
| R1-1, R2-3 | Split belum per pasien | **SEL 5**: split per pasien 4 partisi, verifikasi tanpa overlap pasien/citra, jumlah pasien unik per partisi, `split_manifest.csv` |
| R1-2, R2-3 | Protokol stacking & kebocoran | **SEL 14–16**: backbone dilatih di *train*; checkpoint & bobot FPPR dari *val*; XGBoost dilatih di *meta* (prediksi backbone pada data yang tidak dipakai melatihnya) dengan nested CV per pasien; model final & threshold dipilih di *meta*; *test* hanya sekali (SEL 17). Diagram protokol di **SEL 26** |
| R1-3 | Akurasi val tidak konsisten (Fig. 2 vs Table 3) | **SEL 9**: checkpoint Fase 2 melanjutkan nilai terbaik Fase 1 (`initial_value_threshold`) → nilai di kurva = model yang dimuat. **SEL 14**: akurasi val dihitung ulang dari checkpoint & dicek terhadap history; satu sumber angka (`val_acc_fppr.json`) |
| R1-4 | Bobot FPPR tidak bisa direproduksi | **SEL 14**: bobot dihitung dengan Persamaan (1) dari akurasi val yang sama, dicetak 4 desimal + disimpan |
| R1-5, R2-4 | Level 2 lebih buruk; perlu ablation | **SEL 15–17**: ablation lengkap 14 konfigurasi: 3 backbone, Mean, FPPR, XGB(BB), XGB(BB+FPPR), masing-masing ± TTA. (Catatan: XGBoost versi lama juga terkena bug urutan label train) |
| R2-5 | CI & uji statistik | **SEL 17–18**: bootstrap 95% CI per pasien; DeLong & McNemar + Holm; ΔAUC dengan CI; sensitivitas/spesifisitas pada operating point dari partisi meta |
| R1-6, R2-6 | AUC ≠ kalibrasi | **SEL 19**: Brier, ECE, reliability diagram |
| R1-8 | Tabel SOTA bukan head-to-head | **SEL 22**: tabel literatur diberi label "protokol berbeda", tanpa klaim superioritas. **SEL 25**: baseline klinis CTR otomatis dievaluasi pada split yang identik (head-to-head) |
| R1-7, R2-2 | Klaim cross-domain tanpa data eksternal | **SEL 24**: uji eksternal CheXpert (institusi, labeler, populasi & proporsi AP berbeda) — skenario A (setara) & B (realistis, termasuk komorbid & kelainan non-kardiomegali); subgrup AP vs PA |
| R1-10, R2-7 | Klaim CTR 0.50–0.54 tanpa pengukuran | **SEL 25**: CTR diukur otomatis dengan prosedur terdokumentasi + QC; distribusi CTR per kelompok TP/TN/FP/FN; ekspor 50 citra untuk verifikasi manual radiolog |
| R1-11 | Reprodusibilitas | **SEL 27**: prediksi per sampel, config lengkap (seed, split, aturan checkpoint, bobot FPPR, TTA, hyperparameter XGBoost), environment & `pip freeze` |

**Di luar kode (harus di manuskrip):** judul (R1-7, R2-2), audit sitasi (R1-9), posisi kebaruan (R2-1), serta menulis ulang klaim CTR berdasarkan hasil SEL 25.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## ⚙️ SEL 1 — Install Library

In [ ]:
!pip install -q xgboost shap scikit-learn seaborn matplotlib
!pip install -q tensorflow keras opencv-python-headless pandas numpy pillow
!pip install -q torchxrayvision   # model segmentasi anatomi untuk pengukuran CTR (SEL 25)
print("✅ Semua library siap")

---
## 📦 SEL 2 — Import & Konfigurasi

In [ ]:
import os, json, time, random, warnings, pickle, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from datetime import datetime
warnings.filterwarnings('ignore')

import cv2, glob
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split, StratifiedGroupKFold, GridSearchCV
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from scipy import stats
import itertools, sys, platform, subprocess
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, roc_auc_score, roc_curve, auc,
    precision_score, recall_score, brier_score_loss
)
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
import shap

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input, regularizers
from tensorflow.keras.applications import EfficientNetB4, DenseNet121, ResNet50V2
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_pre
from tensorflow.keras.applications.densenet    import preprocess_input as den_pre
from tensorflow.keras.applications.resnet_v2   import preprocess_input as res_pre
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import tensorflow.keras.backend as K

# TensorFlow tidak memonopoli memori GPU → PyTorch (segmentasi CTR) bisa jalan di GPU yang sama
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g, True)
    except Exception as _e: print('memory growth:', _e)

SEED = 42
np.random.seed(SEED); tf.random.set_seed(SEED); random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

CLASSES   = ['Non_Cardiomegaly', 'Cardiomegaly']
N_CLASSES = 2
PAL = {
    'eff':'#1565C0', 'den':'#C62828', 'res':'#6A1B9A',
    'mean':'#00838F', 'fppr':'#2E7D32', 'xgb':'#E65100', 'xgbf':'#F9A825', 'tta':'#8B0000',
}

plt.rcParams.update({'figure.dpi': 130})
print(f'TensorFlow  : {tf.__version__}')
print(f'GPU         : {tf.config.list_physical_devices("GPU")}')
print(f'Kelas       : {CLASSES}')

---
## ☁️ SEL 3 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

OUTPUT_BASE = '/content/drive/MyDrive/CELM_Cardio_v6'
OUTPUT = OUTPUT_BASE   # di SEL 6 diarahkan ke subfolder per skenario augmentasi
for d in ['models','logs','figures','results']:
    os.makedirs(os.path.join(OUTPUT, d), exist_ok=True)
print(f'✅ Output folder: {OUTPUT}')

---
## 📥 SEL 4 — Download Dataset NIH ChestX-ray14

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# Dataset: NIH ChestX-ray14 (224×224 resized)
# Kaggle : khanfashee/nih-chest-x-ray-14-224x224-resized
# Ukuran : ~42 GB full (atau subset yang lebih kecil)
# Label  : Data_Entry_2017.csv — termasuk "Cardiomegaly" dan "No Finding"
#
# Keunggulan dibanding dataset rahimanshu:
# - 112.120 citra ASLI dari 30.805 pasien NIH Clinical Center
# - Label dianotasi oleh radiolog dan diverifikasi NLP
# - Digunakan sebagai benchmark standar internasional
# ════════════════════════════════════════════════════════════════════════════

from google.colab import files
# files.upload()  # ← uncomment & jalankan untuk upload kaggle.json

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ 2>/dev/null && echo "✅ kaggle.json siap" || echo "⚠️  upload kaggle.json dulu"
!chmod 600 ~/.kaggle/kaggle.json 2>/dev/null
!pip install -q kaggle

NIH_DIR = '/content/nih_raw'
os.makedirs(NIH_DIR, exist_ok=True)

# Download dataset NIH 224x224
print('📥 Download NIH ChestX-ray14 (224×224) ...')
!kaggle datasets download -d khanfashee/nih-chest-x-ray-14-224x224-resized \
    -p {NIH_DIR} --unzip -q

print('\n📁 Isi folder NIH:')
items = sorted(os.listdir(NIH_DIR))
for item in items[:15]:
    p = os.path.join(NIH_DIR, item)
    info = f'({os.path.getsize(p):,} bytes)' if os.path.isfile(p) else '(folder)'
    print(f'   {item}  {info}')

---
## 🗂️ SEL 5 — Filter Dataset & Pemisahan Train / Val / Meta / Test (per Pasien)

In [ ]:
import glob

# ── Cari CSV label ─────────────────────────────────────────────────────────────
csv_candidates = glob.glob(os.path.join(NIH_DIR, '**/*.csv'), recursive=True)
csv_path = None
for f in csv_candidates:
    if any(k in f.lower() for k in ['entry','label','finding']):
        csv_path = f; break
if csv_path is None and csv_candidates:
    csv_path = csv_candidates[0]

print(f'CSV labels: {csv_path}')
df = pd.read_csv(csv_path)
print(f'Total baris: {len(df)}')
print(f'Kolom      : {df.columns.tolist()}')
print(df.head(3))

In [ ]:
# ── Deteksi nama kolom ─────────────────────────────────────────────────────────
img_col   = next((c for c in df.columns if any(k in c.lower()
                  for k in ['image','file','img'])), df.columns[0])
label_col = next((c for c in df.columns if any(k in c.lower()
                  for k in ['finding','label','class','disease'])), df.columns[1])
pid_col   = next((c for c in df.columns if 'patient' in c.lower() and 'id' in c.lower()), None)
assert pid_col is not None, 'Kolom Patient ID tidak ditemukan — wajib untuk split per pasien'

print(f'Image col   : {img_col}')
print(f'Label col   : {label_col}')
print(f'Patient col : {pid_col}')

# Lowercase untuk matching
df['_label_lower'] = df[label_col].str.strip().str.lower()

# ── Filter kelas ───────────────────────────────────────────────────────────────
# Cardiomegaly     = label persis "cardiomegaly" (tanpa penyakit lain)
# Non_Cardiomegaly = label "no finding" (benar-benar normal)
N_PER_CLASS = 3000   # batas maksimum citra per kelas

cardio_df = df[df['_label_lower'] == 'cardiomegaly']
normal_df = df[df['_label_lower'] == 'no finding']

print(f'\nCardiomegaly total : {len(cardio_df):5d} citra | {cardio_df[pid_col].nunique():5d} pasien')
print(f'No Finding total   : {len(normal_df):5d} citra | {normal_df[pid_col].nunique():5d} pasien')

cardio_s = cardio_df.sample(min(N_PER_CLASS, len(cardio_df)), random_state=SEED)
normal_s = normal_df.sample(min(N_PER_CLASS, len(normal_df)), random_state=SEED)
print(f'\nSampled Cardiomegaly    : {len(cardio_s)}')
print(f'Sampled Non-Cardiomegaly: {len(normal_s)}')


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# PEMISAHAN DATA PER PASIEN — 4 PARTISI (R1-1, R1-2, R2-3)
#   train (60%) : melatih ketiga backbone
#   val   (10%) : checkpoint/early stopping + bobot FPPR (Persamaan 1)
#   meta  (15%) : melatih XGBoost meta-learner + memilih model final & threshold
#   test  (15%) : SATU KALI evaluasi akhir — tidak dipakai untuk keputusan apa pun
# Semua citra milik satu pasien berada di satu partisi yang sama.
# Stratifikasi per pasien: pasien yang punya citra Cardiomegaly vs tidak.
# ════════════════════════════════════════════════════════════════════════════

# ── Cari folder yang berisi citra ──────────────────────────────────────────────
img_dirs = []
for root, dirs, flist in os.walk(NIH_DIR):
    if any(f.lower().endswith(('.jpg','.png','.jpeg')) for f in flist):
        img_dirs.append(root)
print(f'Folder citra ditemukan: {len(img_dirs)}')

def find_img(fname, dirs):
    for d in dirs:
        p = os.path.join(d, fname)
        if os.path.exists(p): return p
    return None

SPLITS      = ['train', 'val', 'meta', 'test']
SPLIT_RATIO = {'train': 0.60, 'val': 0.10, 'meta': 0.15, 'test': 0.15}
assert abs(sum(SPLIT_RATIO.values()) - 1) < 1e-9

# ── 1. Gabungkan kedua kelas + cari path citra ────────────────────────────────
sel = pd.concat([cardio_s.assign(cls='Cardiomegaly'),
                 normal_s.assign(cls='Non_Cardiomegaly')], ignore_index=True)
sel['path'] = [find_img(str(f), img_dirs) for f in sel[img_col]]
n_missing = sel['path'].isna().sum()
if n_missing: print(f'⚠️  {n_missing} citra tidak ditemukan di disk — dilewati')
sel = sel.dropna(subset=['path']).reset_index(drop=True)

# ── 2. Split pada level PASIEN (stratified) ───────────────────────────────────
pat = (sel.groupby(pid_col)['cls']
          .apply(lambda s: int((s == 'Cardiomegaly').any()))
          .rename('has_cardio').reset_index())

def _strat(df_, frac):
    return train_test_split(df_, test_size=frac, random_state=SEED, stratify=df_['has_cardio'])

r = SPLIT_RATIO
rest,   pat_te = _strat(pat,  r['test'])
rest,   pat_me = _strat(rest, r['meta'] / (1 - r['test']))
pat_tr, pat_va = _strat(rest, r['val']  / (r['train'] + r['val']))

split_of = {}
for name, part in [('train', pat_tr), ('val', pat_va), ('meta', pat_me), ('test', pat_te)]:
    split_of.update({p: name for p in part[pid_col]})
sel['split'] = sel[pid_col].map(split_of)

# ── 3. Salin ke DATA_DIR/{split}/{kelas} ──────────────────────────────────────
DATA_DIR = '/content/celm_data'
if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)            # bersihkan sisa run sebelumnya
for s in SPLITS:
    for cls in CLASSES:
        os.makedirs(os.path.join(DATA_DIR, s, cls), exist_ok=True)

print('\n📋 Menyalin citra ...')
for p_src, s, cls in zip(sel['path'], sel['split'], sel['cls']):
    shutil.copy(p_src, os.path.join(DATA_DIR, s, cls, os.path.basename(p_src)))

TRAIN_DIR = os.path.join(DATA_DIR, 'train')
VAL_DIR   = os.path.join(DATA_DIR, 'val')
META_DIR  = os.path.join(DATA_DIR, 'meta')
TEST_DIR  = os.path.join(DATA_DIR, 'test')
SPLIT_DIR = {'train': TRAIN_DIR, 'val': VAL_DIR, 'meta': META_DIR, 'test': TEST_DIR}

# ── 4. Verifikasi: tidak ada pasien / citra yang bocor antar partisi ──────────
pids = {s: set(sel.loc[sel['split']==s, pid_col]) for s in SPLITS}
imgs = {s: set(sel.loc[sel['split']==s, img_col]) for s in SPLITS}
for a, b in itertools.combinations(SPLITS, 2):
    assert not (pids[a] & pids[b]), f'Patient leakage {a}↔{b}: {len(pids[a] & pids[b])} pasien'
    assert not (imgs[a] & imgs[b]), f'Image leakage {a}↔{b}'
for s in SPLITS:
    for cls in CLASSES:
        n_disk = len(os.listdir(os.path.join(DATA_DIR, s, cls)))
        n_man  = ((sel['split']==s) & (sel['cls']==cls)).sum()
        assert n_disk == n_man, f'{s}/{cls}: disk={n_disk} manifest={n_man}'
print('✅ Tidak ada patient/image leakage antar train-val-meta-test')

# Peta nama file → Patient ID (dipakai untuk CV & bootstrap per pasien)
IMG2PID = dict(zip(sel[img_col].astype(str).map(os.path.basename), sel[pid_col]))

# ── 5. Ringkasan & manifest ───────────────────────────────────────────────────
summary = (sel.pivot_table(index='cls', columns='split', values=img_col,
                           aggfunc='count', fill_value=0)
              .reindex(index=CLASSES, columns=SPLITS, fill_value=0))
summary.loc['Total'] = summary.sum()
summary['Total'] = summary.sum(axis=1)
pat_cnt = sel.groupby('split')[pid_col].nunique().reindex(SPLITS)
pat_cls = (sel.groupby(['split','cls'])[pid_col].nunique().unstack()
              .reindex(index=SPLITS, columns=CLASSES).fillna(0).astype(int))

print('\n📊 Jumlah citra per partisi:')
print(summary.to_string())
print('\n👤 Jumlah pasien unik per partisi (per kelas citra):')
print(pd.concat([pat_cls, pat_cnt.rename('Total')], axis=1).to_string())
print('\n📐 Proporsi Cardiomegaly per partisi:')
for s in SPLITS:
    print(f'   {s:5s}: {summary.loc["Cardiomegaly", s] / summary.loc["Total", s] * 100:.1f}%')

SPLIT_COUNTS = {s: {'images': int(summary.loc['Total', s]),
                    'images_cardiomegaly': int(summary.loc['Cardiomegaly', s]),
                    'images_non_cardiomegaly': int(summary.loc['Non_Cardiomegaly', s]),
                    'patients': int(pat_cnt[s])} for s in SPLITS}

os.makedirs(os.path.join(OUTPUT, 'results'), exist_ok=True)
sel[[img_col, pid_col, label_col, 'cls', 'split']].to_csv(
    os.path.join(OUTPUT, 'results', 'split_manifest.csv'), index=False)
print(f'\n💾 Manifest: {os.path.join(OUTPUT, "results", "split_manifest.csv")}')

---
## ⚙️ SEL 6 — Hyperparameter & Protokol

In [ ]:
IMG_SIZE    = (224, 224)
BATCH       = 32

# ── Learning Rate ──────────────────────────────────────────────────────────────
LR_HEAD     = 1e-3    # Fase 1: head saja (frozen backbone)
LR_FINE     = 2e-5    # Fase 2: Cosine Decay dari nilai ini

# ── Epoch ──────────────────────────────────────────────────────────────────────
EPOCHS_HEAD = 15      # warm-up head (EarlyStopping patience=5)
EPOCHS_FINE = 60      # fine-tune (EarlyStopping patience=12)

# ── Skenario augmentasi (ablation study) ──────────────────────────────────────
#   'none'   : tanpa augmentasi training
#   'flip'   : augmentasi geometri/intensitas + horizontal flip (setting lama)
#   'noflip' : augmentasi geometri/intensitas TANPA horizontal flip (disarankan:
#              flip memindahkan jantung ke kanan → anatomi tidak realistis)
AUG_MODE = 'noflip'
assert AUG_MODE in ('none', 'flip', 'noflip')

# Model, log & hasil tiap skenario disimpan terpisah agar bisa dibandingkan
OUTPUT = os.path.join(OUTPUT_BASE, f'aug_{AUG_MODE}')
for d in ['models','logs','figures','results']:
    os.makedirs(os.path.join(OUTPUT, d), exist_ok=True)

# ── Protokol evaluasi (R1-2, R2-5) ───────────────────────────────────────────
TTA_N       = 5        # jumlah augmentasi saat TTA
TTA_SEED    = SEED     # seed TTA → hasil TTA bisa direproduksi
CV_FOLDS    = 5        # fold (per pasien) untuk XGBoost di partisi meta
XGB_GRID    = {        # grid hyperparameter XGBoost (dipilih via CV di partisi meta)
    'max_depth'       : [2, 3, 4],
    'n_estimators'    : [100, 300],
    'learning_rate'   : [0.05],
    'min_child_weight': [1, 5],
}
N_BOOT      = 2000     # resample bootstrap (per pasien) untuk 95% CI
ECE_BINS    = 10

# Salin manifest split ke folder skenario
shutil.copy(os.path.join(OUTPUT_BASE, 'results', 'split_manifest.csv'),
            os.path.join(OUTPUT, 'results', 'split_manifest.csv'))

print('⚙️  Konfigurasi v7:')
cfg_items = [('IMG_SIZE',IMG_SIZE),('BATCH',BATCH),
             ('LR_HEAD',LR_HEAD),('LR_FINE',LR_FINE),
             ('EPOCHS_HEAD',EPOCHS_HEAD),('EPOCHS_FINE',EPOCHS_FINE),
             ('AUG_MODE',AUG_MODE),('TTA_N',TTA_N),('CV_FOLDS',CV_FOLDS),
             ('N_BOOT',N_BOOT),('OUTPUT',OUTPUT)]
for k,v in cfg_items:
    print(f'   {k:<15}: {v}')

---
## 🎯 SEL 7 — Focal Loss, Generator & TTA (ber-seed)

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# FOCAL LOSS
# Menambahkan bobot (1-p)^gamma pada Cross-Entropy sehingga sampel
# yang susah diklasifikasi mendapat perhatian lebih besar.
# Terbukti meningkatkan akurasi pada dataset medis yang imbalanced.
# ════════════════════════════════════════════════════════════════════════════
def focal_loss(gamma=2.0, alpha=0.25):
    def _fl(y_true, y_pred):
        eps   = K.epsilon()
        y_pred = K.clip(y_pred, eps, 1.0 - eps)
        ce    = -y_true * K.log(y_pred)
        fl    = alpha * K.pow(1.0 - y_pred, gamma) * ce
        return K.sum(fl, axis=1)
    return _fl

FL = focal_loss(gamma=2.0, alpha=0.25)   # instance untuk custom_objects


# ════════════════════════════════════════════════════════════════════════════
# GENERATOR — preprocessing_function backbone-specific
# ════════════════════════════════════════════════════════════════════════════
def make_train_gen(preproc_fn, img_size=IMG_SIZE, batch=BATCH):
    if AUG_MODE == 'none':
        train_dg = ImageDataGenerator(preprocessing_function=preproc_fn)
    else:
        train_dg = ImageDataGenerator(
            preprocessing_function = preproc_fn,
            rotation_range         = 15,
            width_shift_range      = 0.10,
            height_shift_range     = 0.10,
            zoom_range             = 0.15,
            horizontal_flip        = (AUG_MODE == 'flip'),
            shear_range            = 0.08,
            brightness_range       = [0.85, 1.15],
            fill_mode              = 'nearest',
        )
    return train_dg.flow_from_directory(TRAIN_DIR, target_size=img_size, class_mode='categorical',
                                        classes=CLASSES, batch_size=batch, shuffle=True, seed=SEED)


def eval_gen(preproc_fn, data_dir, img_size=IMG_SIZE, batch=BATCH):
    """Generator evaluasi: TANPA augmentasi, TANPA shuffle (urutan file tetap)."""
    dg = ImageDataGenerator(preprocessing_function=preproc_fn)
    return dg.flow_from_directory(data_dir, target_size=img_size, class_mode='categorical',
                                  classes=CLASSES, batch_size=batch, shuffle=False)


# ════════════════════════════════════════════════════════════════════════════
# TEST-TIME AUGMENTATION (TTA) — ber-seed agar bisa direproduksi (R1-11)
# Rata-rata prediksi dari TTA_N versi augmentasi ringan.
# ════════════════════════════════════════════════════════════════════════════
TTA_TRANSFORMS = dict(rotation_range=10, zoom_range=0.05, horizontal_flip=(AUG_MODE == 'flip'))

def tta_predict(model, preproc_fn, data_dir, img_size=IMG_SIZE, batch=BATCH,
                n=TTA_N, seed=TTA_SEED):
    tta_dg = ImageDataGenerator(preprocessing_function=preproc_fn, **TTA_TRANSFORMS)
    preds_list = []
    for k in range(n):
        np.random.seed(seed + k); random.seed(seed + k)
        gen = tta_dg.flow_from_directory(
            data_dir, target_size=img_size, batch_size=batch, class_mode='categorical',
            shuffle=False, classes=CLASSES, seed=seed + k)
        preds_list.append(model.predict(gen, verbose=0))
    return np.mean(preds_list, axis=0)


print(f'✅ Focal Loss · Generator · TTA siap  (AUG_MODE={AUG_MODE}, TTA_N={TTA_N})')


---
## 🏗️ SEL 8 — Arsitektur Model (Head Lebih Dalam)

In [ ]:
def build_model(bb_name, lr=1e-3, freeze=True):
    """
    Arsitektur v6: Head lebih dalam untuk kapasitas lebih besar.
    Backbone → GAP → BN
             → Dense(1024,relu) → BN → Drop(0.40)
             → Dense(512, relu) → BN → Drop(0.30)
             → Dense(256, relu) → Drop(0.20)
             → Softmax(2)
    """
    inp  = Input(shape=(*IMG_SIZE, 3), name='input_cxr')
    BB   = {'EfficientNetB4': EfficientNetB4,
            'DenseNet121'   : DenseNet121,
            'ResNet50V2'    : ResNet50V2}

    base = BB[bb_name](include_top=False, weights='imagenet', input_tensor=inp)
    base.trainable = not freeze

    x = layers.GlobalAveragePooling2D(name='gap')(base.output)
    x = layers.BatchNormalization(name='bn_top')(x)

    x = layers.Dense(1024, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-5), name='fc1')(x)
    x = layers.BatchNormalization(name='bn1')(x)
    x = layers.Dropout(0.40, name='drop1')(x)

    x = layers.Dense(512, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-5), name='fc2')(x)
    x = layers.BatchNormalization(name='bn2')(x)
    x = layers.Dropout(0.30, name='drop2')(x)

    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-5), name='fc3')(x)
    x = layers.Dropout(0.20, name='drop3')(x)

    out = layers.Dense(N_CLASSES, activation='softmax', name='output')(x)
    model = Model(inp, out, name=f'CELMv6_{bb_name}')
    model.compile(
        optimizer = Adam(lr),
        loss      = FL,   # Focal Loss
        metrics   = ['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    return model, base

print('✅ build_model() siap')
print('   Head: Dense(1024→512→256) dengan L2 + Dropout')
print('   Loss: Focal Loss (gamma=2.0, alpha=0.25)')

---
## 🔧 SEL 9 — Fungsi Training (2 Fase + Cosine Decay, checkpoint konsisten)

In [ ]:
def train_backbone(bb_name, preproc_fn, model_path, log_path, cls_wt):
    """
    Training 2 fase. Hanya partisi TRAIN untuk fitting, hanya VAL untuk checkpoint.
    Fase 1: head saja (backbone frozen). Fase 2: full fine-tune, Cosine Decay LR.

    Aturan checkpoint (R1-3): file checkpoint = bobot dengan val_accuracy tertinggi
    di GABUNGAN Fase 1 + Fase 2. ModelCheckpoint Fase 2 diberi
    initial_value_threshold = val_acc terbaik Fase 1, sehingga epoch Fase 2 yang
    lebih buruk TIDAK menimpa checkpoint Fase 1 (bug versi lama).
    """
    tr = make_train_gen(preproc_fn)
    va = eval_gen(preproc_fn, VAL_DIR)

    # ── FASE 1: Head warm-up ─────────────────────────────────────────────────
    model, base = build_model(bb_name, lr=LR_HEAD, freeze=True)
    n_tr = sum(v.shape.num_elements() for v in model.trainable_variables)
    print(f'\n🚀 {bb_name} — FASE 1 (head only, {n_tr:,} params)')
    cbs1 = [
        EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
        ModelCheckpoint(model_path, monitor='val_accuracy', mode='max',
                        save_best_only=True, verbose=1),
        CSVLogger(log_path.replace('.csv', '_fase1.csv')),
    ]
    h1 = model.fit(tr, epochs=EPOCHS_HEAD, validation_data=va,
                   callbacks=cbs1, class_weight=cls_wt, verbose=1)
    acc_h1 = max(h1.history['val_accuracy'])
    n_ep1  = len(h1.history['val_accuracy'])
    print(f'   Fase 1 best val_acc = {acc_h1:.4f}')

    # ── FASE 2: Full fine-tune dengan Cosine Decay LR ────────────────────────
    print(f'\n🔧 {bb_name} — FASE 2 (full fine-tune, Cosine Decay LR)')
    base.trainable = True
    total_steps = EPOCHS_FINE * len(tr)
    lr_cos = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=LR_FINE, decay_steps=total_steps, alpha=1e-7)
    model.compile(optimizer=Adam(lr_cos), loss=FL,
                  metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
    cbs2 = [
        EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1),
        ModelCheckpoint(model_path, monitor='val_accuracy', mode='max', save_best_only=True,
                        initial_value_threshold=acc_h1, verbose=1),   # ← perbaikan R1-3
        CSVLogger(log_path),
    ]
    h2 = model.fit(tr, epochs=EPOCHS_FINE, validation_data=va,
                   callbacks=cbs2, class_weight=cls_wt, verbose=1)

    # Gabung history
    hist = {k: list(h1.history[k]) + list(h2.history.get(k, [])) for k in h1.history}
    best_acc = max(hist['val_accuracy'])
    best_ep  = hist['val_accuracy'].index(best_acc) + 1     # epoch pertama yang mencapai nilai tsb
    ckpt = {'backbone': bb_name, 'val_accuracy_history': float(best_acc),
            'epoch': int(best_ep), 'phase': 1 if best_ep <= n_ep1 else 2,
            'phase1_epochs': int(n_ep1), 'total_epochs': len(hist['val_accuracy']),
            'rule': 'max val_accuracy over phase 1 + phase 2 (val partition only)'}
    print(f'\n✅ {bb_name} checkpoint: val_acc = {best_acc:.4f} '
          f'(epoch {best_ep}, fase {ckpt["phase"]})')
    return model, base, hist, ckpt

print('✅ train_backbone() siap')

---
## 🏋️ SEL 10 — Training EfficientNetB4

In [ ]:
# Class weight dari partisi train
from pathlib import Path as Pth
y_all = []
for ci, cls in enumerate(CLASSES):
    d = os.path.join(TRAIN_DIR, cls)
    y_all.extend([ci] * len([f for f in os.listdir(d)
                              if Pth(f).suffix.lower() in {'.jpg','.png','.jpeg'}]))
cw  = compute_class_weight('balanced', classes=np.array([0,1]), y=np.array(y_all))
CLS_WT = {0: float(cw[0]), 1: float(cw[1])}
print(f'Class weight: {CLS_WT}')

EFF_PATH = os.path.join(OUTPUT, 'models', 'eff_v7.keras')
LOG_EFF  = os.path.join(OUTPUT, 'logs',   'eff_v7.csv')
model_eff, base_eff, hist_eff, ckpt_eff = train_backbone(
    'EfficientNetB4', eff_pre, EFF_PATH, LOG_EFF, CLS_WT)

---
## 🏋️ SEL 11 — Training DenseNet121

In [ ]:
DEN_PATH = os.path.join(OUTPUT, 'models', 'den_v7.keras')
LOG_DEN  = os.path.join(OUTPUT, 'logs',   'den_v7.csv')
model_den, base_den, hist_den, ckpt_den = train_backbone(
    'DenseNet121', den_pre, DEN_PATH, LOG_DEN, CLS_WT)

---
## 🏋️ SEL 12 — Training ResNet50V2

In [ ]:
RES_PATH = os.path.join(OUTPUT, 'models', 'res_v7.keras')
LOG_RES  = os.path.join(OUTPUT, 'logs',   'res_v7.csv')
model_res, base_res, hist_res, ckpt_res = train_backbone(
    'ResNet50V2', res_pre, RES_PATH, LOG_RES, CLS_WT)

---
## 📊 SEL 13 — Training History

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Training History — CELM-Cardio v7 (train vs val partition)\n'
             'Garis putus hijau = epoch checkpoint (model yang dipakai)',
             fontsize=12, fontweight='bold')

for ax, hist, ck, col, name in [
    (axes[0], hist_eff, ckpt_eff, PAL['eff'], 'EfficientNetB4'),
    (axes[1], hist_den, ckpt_den, PAL['den'], 'DenseNet121'),
    (axes[2], hist_res, ckpt_res, PAL['res'], 'ResNet50V2'),
]:
    ep = range(1, len(hist['val_accuracy'])+1)
    ax.plot(ep, hist['accuracy'],     color=col, lw=2.0, alpha=0.6, label='Train')
    ax.plot(ep, hist['val_accuracy'], color=col, lw=2.5, label='Val')
    ax.axvline(ck['phase1_epochs'] + 0.5, color='grey', ls=':', lw=1.2, label='Fase 1 → 2')
    ax.axvline(ck['epoch'], color='green', ls='--', lw=1.5, label='Checkpoint')
    ax.scatter([ck['epoch']], [ck['val_accuracy_history']], color='green', zorder=5)
    ax.set_title(f'{name}\nCheckpoint val_acc = {ck["val_accuracy_history"]*100:.2f}% '
                 f'(epoch {ck["epoch"]}, fase {ck["phase"]})', fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_ylim(0.4, 1.02)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT,'figures','history.png'), dpi=300, bbox_inches='tight')
plt.show()

---
## ⚖️ SEL 14 — Prediksi Backbone (val · meta · test) & Bobot FPPR

Checkpoint dimuat ulang. **Akurasi val** dihitung sekali dari checkpoint dan dicocokkan dengan nilai di kurva training (R1-3). **Bobot FPPR** (Persamaan 1) dihitung dari akurasi val yang sama (R1-4):

$$w_k = \frac{\text{Acc}^{val}_k}{\sum_j \text{Acc}^{val}_j}$$

In [ ]:
# Bebaskan model hasil training (bobot terbaik sudah ada di file checkpoint) → hemat RAM/GPU
import gc
for _v in ['model_eff', 'model_den', 'model_res', 'base_eff', 'base_den', 'base_res']:
    globals().pop(_v, None)
tf.keras.backend.clear_session(); gc.collect()

CO = {'_fl': FL}   # custom_objects untuk Focal Loss
BACKBONES = [  # (nama, key, path, preprocess)
    ('EfficientNetB4', 'eff', EFF_PATH, eff_pre),
    ('DenseNet121',    'den', DEN_PATH, den_pre),
    ('ResNet50V2',     'res', RES_PATH, res_pre),
]
BB_KEYS = ['eff', 'den', 'res']
CKPT    = {'eff': ckpt_eff, 'den': ckpt_den, 'res': ckpt_res}
MODELS  = {k: keras.models.load_model(p, custom_objects=CO) for _, k, p, _ in BACKBONES}
model_eff, model_den, model_res = MODELS['eff'], MODELS['den'], MODELS['res']
print('✅ Checkpoint dimuat')

# ── Prediksi: val (plain), meta & test (plain + TTA) ──────────────────────────
EVAL_SPLITS = ['val', 'meta', 'test']
P = {k: {s: {} for s in EVAL_SPLITS} for k in BB_KEYS}   # P[key][split][mode] → (n,2)
FILES, Y = {}, {}
for name, k, _, pre in BACKBONES:
    for s in EVAL_SPLITS:
        g = eval_gen(pre, SPLIT_DIR[s])
        P[k][s]['plain'] = MODELS[k].predict(g, verbose=0)
        if s in ('meta', 'test'):
            P[k][s]['tta'] = tta_predict(MODELS[k], pre, SPLIT_DIR[s])
        if s not in FILES:
            FILES[s], Y[s] = list(g.filenames), np.asarray(g.classes)
        else:
            assert FILES[s] == list(g.filenames), 'Urutan file berbeda antar backbone!'
    print(f'   {name}: prediksi selesai')

y_va, y_me, y_te = Y['val'], Y['meta'], Y['test']
PID = {s: np.array([IMG2PID[os.path.basename(f)] for f in FILES[s]]) for s in EVAL_SPLITS}
print(f'\nval={len(y_va)}  meta={len(y_me)}  test={len(y_te)} citra')

# ── Akurasi val: satu sumber angka (R1-3) ─────────────────────────────────────
VAL_ACC = {k: float(accuracy_score(y_va, P[k]['val']['plain'].argmax(1))) for k in BB_KEYS}
print('\nAkurasi val (checkpoint yang dimuat)  vs  nilai di kurva training:')
for name, k, _, _ in BACKBONES:
    h = CKPT[k]['val_accuracy_history']
    flag = '✅' if abs(VAL_ACC[k] - h) < 1e-3 else '⚠️  BEDA — cek checkpoint'
    print(f'   {name:15s}: {VAL_ACC[k]*100:6.2f}%   vs   {h*100:6.2f}%   {flag}')

# ── Bobot FPPR — Persamaan (1) (R1-4) ─────────────────────────────────────────
tot = sum(VAL_ACC.values())
W = {k: VAL_ACC[k] / tot for k in BB_KEYS}
W_EFF, W_DEN, W_RES = W['eff'], W['den'], W['res']
print(f'\n⚖️  FPPR (Persamaan 1):  Σ Acc_val = {tot*100:.2f}')
for name, k, _, _ in BACKBONES:
    print(f'   w_{k} = {VAL_ACC[k]*100:.2f} / {tot*100:.2f} = {W[k]:.4f}')

json.dump({'val_accuracy': VAL_ACC, 'fppr_weights': W,
           'checkpoint': CKPT, 'n_val_images': int(len(y_va))},
          open(os.path.join(OUTPUT, 'results', 'val_acc_fppr.json'), 'w'), indent=2)
print(f'\n💾 {os.path.join(OUTPUT, "results", "val_acc_fppr.json")}')

---
## 🌲 SEL 15 — Level 1 (Mean, FPPR) & Level 2 (XGBoost) — Ablation

* **XGBoost dilatih hanya pada partisi meta.** Fitur meta adalah prediksi backbone untuk citra yang **tidak pernah** dipakai melatih backbone tersebut maupun untuk checkpoint-nya.
* Hyperparameter XGBoost dipilih dengan *StratifiedGroupKFold* per pasien di dalam partisi meta.
* Prediksi XGBoost pada partisi meta untuk pemilihan model dibuat dengan **nested CV** (out-of-fold), sehingga tidak bias.
* Varian: **XGB(BB)** = 6 fitur probabilitas backbone; **XGB(BB+FPPR)** = 8 fitur (seperti manuskrip). Versi TTA dilatih terpisah pada fitur TTA.
* Partisi test **tidak** dipakai di sel ini kecuali untuk menghasilkan prediksi (tanpa evaluasi).

In [ ]:
BASE_CFGS = ['EfficientNetB4', 'DenseNet121', 'ResNet50V2', 'Mean', 'FPPR', 'XGB(BB)', 'XGB(BB+FPPR)']
CONFIGS   = BASE_CFGS + [c + ' +TTA' for c in BASE_CFGS]

FEAT_BB  = ['Eff-Non','Eff-Card','Den-Non','Den-Card','Res-Non','Res-Card']
FEAT_BBF = FEAT_BB + ['FPPR-Non','FPPR-Card']

def simple_ensembles(Pe, Pd, Pr):
    """Probabilitas kelas Cardiomegaly untuk backbone tunggal & ensemble Level 1."""
    fppr = W_EFF * Pe + W_DEN * Pd + W_RES * Pr
    return {'EfficientNetB4': Pe[:, 1], 'DenseNet121': Pd[:, 1], 'ResNet50V2': Pr[:, 1],
            'Mean': ((Pe + Pd + Pr) / 3)[:, 1], 'FPPR': fppr[:, 1]}

def meta_features(Pe, Pd, Pr, with_fppr):
    X = [Pe, Pd, Pr]
    if with_fppr:
        X.append(W_EFF * Pe + W_DEN * Pd + W_RES * Pr)
    return np.concatenate(X, axis=1)

def make_xgb(y):
    spw = float((y == 0).sum()) / max(float((y == 1).sum()), 1)
    return xgb.XGBClassifier(subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw,
                             eval_metric='logloss', random_state=SEED, n_jobs=-1)

def fit_xgb_cv(X, y, groups):
    """GridSearch per pasien → model final (refit di seluruh meta) + prediksi nested-CV OOF."""
    outer = StratifiedGroupKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    inner = StratifiedGroupKFold(n_splits=CV_FOLDS - 1, shuffle=True, random_state=SEED)
    oof = np.zeros(len(y))
    for tr_i, te_i in outer.split(X, y, groups):
        gs_in = GridSearchCV(make_xgb(y[tr_i]), XGB_GRID, scoring='roc_auc', cv=inner, n_jobs=1)
        gs_in.fit(X[tr_i], y[tr_i], groups=groups[tr_i])
        oof[te_i] = gs_in.predict_proba(X[te_i])[:, 1]
    gs = GridSearchCV(make_xgb(y), XGB_GRID, scoring='roc_auc', cv=outer, n_jobs=1, refit=True)
    gs.fit(X, y, groups=groups)
    return gs.best_estimator_, gs.best_params_, float(gs.best_score_), oof

PROB = {'meta': {}, 'test': {}}        # PROB[split][config] → prob Cardiomegaly
XGB_MODELS, XGB_INFO, META_X = {}, {}, {}

for mode in ['plain', 'tta']:
    sfx = '' if mode == 'plain' else ' +TTA'
    for s in ['meta', 'test']:
        for name, p in simple_ensembles(*(P[k][s][mode] for k in BB_KEYS)).items():
            PROB[s][name + sfx] = p
    for xname, wf in [('XGB(BB)', False), ('XGB(BB+FPPR)', True)]:
        cfg = xname + sfx
        Xm = meta_features(*(P[k]['meta'][mode] for k in BB_KEYS), wf)
        Xt = meta_features(*(P[k]['test'][mode] for k in BB_KEYS), wf)
        t0 = time.time()
        model, params, cv_auc, oof = fit_xgb_cv(Xm, y_me, PID['meta'])
        PROB['meta'][cfg] = oof                         # out-of-fold (nested CV)
        PROB['test'][cfg] = model.predict_proba(Xt)[:, 1]
        XGB_MODELS[cfg], META_X[cfg] = model, (Xm, Xt)
        XGB_INFO[cfg] = {'n_features': int(Xm.shape[1]), 'best_params': params,
                         'cv_auc_meta': cv_auc, 'trained_on': 'meta partition only',
                         'n_train': int(len(y_me))}
        with open(os.path.join(OUTPUT, 'models', f'xgb_{xname.replace("(","_").replace(")","").replace("+","_")}{"_tta" if mode=="tta" else ""}.pkl'), 'wb') as f:
            pickle.dump(model, f)
        print(f'🌲 {cfg:22s} params={params}  CV-AUC(meta)={cv_auc:.4f}  ({time.time()-t0:.0f}s)')

assert set(PROB['meta']) == set(PROB['test']) == set(CONFIGS)
print('\n✅ 14 konfigurasi siap. Partisi test belum dievaluasi.')

---
## 🎯 SEL 16 — Pemilihan Model Final & Operating Point di Partisi Meta

Model final = konfigurasi dengan **AUC tertinggi di partisi meta** (untuk XGBoost memakai prediksi nested-CV out-of-fold); jika seri (4 desimal), dipilih **Brier score meta terendah**. Threshold operasional = **Youden J** di partisi meta. Keduanya ditetapkan **sebelum** test dievaluasi.

In [ ]:
def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    j = np.argmax(tpr - fpr)
    return float(np.clip(thr[j], 0.0, 1.0))

META_AUC = {c: float(roc_auc_score(y_me, PROB['meta'][c])) for c in CONFIGS}
THR      = {c: youden_threshold(y_me, PROB['meta'][c]) for c in CONFIGS}
META_BRIER = {c: float(brier_score_loss(y_me, PROB['meta'][c])) for c in CONFIGS}
# Kriteria: AUC meta tertinggi (dibulatkan 4 desimal); seri → Brier meta terendah
FINAL_MODEL = max(CONFIGS, key=lambda c: (round(META_AUC[c], 4), -META_BRIER[c]))

print('AUC di partisi META (dasar pemilihan model):')
for c in sorted(CONFIGS, key=lambda c: (round(META_AUC[c], 4), -META_BRIER[c]), reverse=True):
    star = '★ FINAL' if c == FINAL_MODEL else ''
    print(f'   {c:22s}  AUC={META_AUC[c]:.4f}  Brier={META_BRIER[c]:.4f}  thr(Youden)={THR[c]:.3f}  {star}')
print(f'\n🔒 Model final dikunci: {FINAL_MODEL!r} (threshold {THR[FINAL_MODEL]:.3f})')

---
## 🧪 SEL 17 — Evaluasi Akhir di Test (Satu Kali) + 95% CI

* Semua 14 konfigurasi dievaluasi **sekali** pada test. Metrik utama memakai threshold 0.5; kolom `@thr` memakai threshold Youden dari partisi meta.
* 95% CI dari **bootstrap per pasien** (resample pasien, bukan citra, karena citra satu pasien saling berkorelasi). Resample yang sama dipakai untuk semua konfigurasi (berpasangan).
* AUC mengukur **diskriminasi**; kalibrasi diukur dengan **Brier** dan **ECE**.

In [ ]:
def ece_score(y, p, n_bins=ECE_BINS):
    bins = np.linspace(0, 1, n_bins + 1)
    idx  = np.clip(np.digitize(p, bins[1:-1]), 0, n_bins - 1)
    ece  = 0.0
    for b in range(n_bins):
        m = idx == b
        if m.any():
            ece += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(ece)

def point_metrics(y, p, thr=0.5):
    yh = (p >= thr).astype(int)
    tp = int(((yh == 1) & (y == 1)).sum()); tn = int(((yh == 0) & (y == 0)).sum())
    fp = int(((yh == 1) & (y == 0)).sum()); fn = int(((yh == 0) & (y == 1)).sum())
    sens = tp / max(tp + fn, 1); spec = tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    f1   = 2 * prec * sens / max(prec + sens, 1e-12)
    return {'AUC': float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else np.nan,
            'Accuracy': (tp + tn) / len(y), 'Sensitivity': sens, 'Specificity': spec,
            'Precision': prec, 'F1': f1,
            'Brier': float(brier_score_loss(y, p)), 'ECE': ece_score(y, p)}

def make_boot_idx(y, groups, n_boot=N_BOOT, seed=SEED):
    """Indeks bootstrap per pasien (cluster bootstrap); dipakai ulang untuk semua model."""
    rng = np.random.default_rng(seed)
    ug  = np.unique(groups)
    by_g = {g: np.where(groups == g)[0] for g in ug}
    out = []
    while len(out) < n_boot:
        gs  = rng.choice(ug, size=len(ug), replace=True)
        idx = np.concatenate([by_g[g] for g in gs])
        if len(np.unique(y[idx])) == 2:
            out.append(idx)
    return out

BOOT_IDX = make_boot_idx(y_te, PID['test'])
METRICS  = ['AUC', 'Accuracy', 'Sensitivity', 'Specificity', 'Precision', 'F1', 'Brier', 'ECE']

EVAL, BOOT = {}, {}
t0 = time.time()
for c in CONFIGS:
    p = PROB['test'][c]
    pm = point_metrics(y_te, p)
    bs = np.array([[point_metrics(y_te[i], p[i])[m] for m in METRICS] for i in BOOT_IDX])
    BOOT[c] = dict(zip(METRICS, bs.T))
    ci = {m: (float(np.percentile(BOOT[c][m], 2.5)), float(np.percentile(BOOT[c][m], 97.5)))
          for m in METRICS}
    op = point_metrics(y_te, p, THR[c])
    EVAL[c] = {'point': pm, 'ci': ci, 'thr': THR[c], 'op': op,
               'y_pred': (p >= 0.5).astype(int), 'p': p}
print(f'Bootstrap {len(BOOT_IDX)} resample per pasien selesai ({time.time()-t0:.0f}s)')

def fmt_ci(c, m, pct=False):
    v, (lo, hi) = EVAL[c]['point'][m], EVAL[c]['ci'][m]
    return f'{v*100:.2f} [{lo*100:.2f}–{hi*100:.2f}]' if pct else f'{v:.3f} [{lo:.3f}–{hi:.3f}]'

rows = []
for c in CONFIGS:
    e = EVAL[c]
    rows.append({
        'Model'          : c + (' ★' if c == FINAL_MODEL else ''),
        'AUC [95% CI]'   : fmt_ci(c, 'AUC'),
        'Acc % [95% CI]' : fmt_ci(c, 'Accuracy', True),
        'Sens % [95% CI]': fmt_ci(c, 'Sensitivity', True),
        'Spec % [95% CI]': fmt_ci(c, 'Specificity', True),
        'F1'             : f"{e['point']['F1']:.3f}",
        'Brier'          : f"{e['point']['Brier']:.3f}",
        'ECE'            : f"{e['point']['ECE']:.3f}",
        'thr(meta)'      : f"{e['thr']:.3f}",
        'Sens@thr %'     : f"{e['op']['Sensitivity']*100:.2f}",
        'Spec@thr %'     : f"{e['op']['Specificity']*100:.2f}",
    })
df_res = pd.DataFrame(rows)
pd.set_option('display.width', 250)
print('\n' + '═'*120)
print('📊 HASIL TEST (evaluasi tunggal) — threshold 0.5 kecuali kolom @thr')
print('═'*120)
print(df_res.to_string(index=False))

# CSV numerik (untuk tabel manuskrip & ablation augmentasi)
num = []
for c in CONFIGS:
    e = EVAL[c]; r_ = {'Model': c, 'Final': c == FINAL_MODEL, 'AUC_meta': META_AUC[c]}
    for m in METRICS:
        r_[m] = e['point'][m]; r_[m + '_lo'], r_[m + '_hi'] = e['ci'][m]
    r_.update({'Threshold_meta': e['thr'], 'Sensitivity_at_thr': e['op']['Sensitivity'],
               'Specificity_at_thr': e['op']['Specificity']})
    num.append(r_)
df_num = pd.DataFrame(num)
df_num.to_csv(os.path.join(OUTPUT, 'results', 'final_metrics.csv'), index=False)
df_res.to_csv(os.path.join(OUTPUT, 'results', 'final_metrics_formatted.csv'), index=False)

print(f'\n📋 Classification report — model final ({FINAL_MODEL}, threshold 0.5):')
print(classification_report(y_te, EVAL[FINAL_MODEL]['y_pred'], target_names=CLASSES, digits=4))

---
## 📐 SEL 18 — Uji Statistik Berpasangan (R2-5)

* **DeLong** untuk perbedaan AUC, **McNemar exact** untuk perbedaan accuracy (threshold 0.5), p-value dikoreksi **Holm**.
* **ΔAUC [95% CI]** dari bootstrap per pasien yang sama (lebih tahan terhadap korelasi antar-citra satu pasien daripada DeLong).
* Perbandingan: model final vs semua konfigurasi lain, setiap konfigurasi vs versi TTA-nya, dan ablation kunci (FPPR vs Mean, XGBoost vs Level 1).

In [ ]:
# ── DeLong (Sun & Xu, 2014) ───────────────────────────────────────────────────
def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1; i = j
    T2 = np.empty(N); T2[J] = T
    return T2

def delong_test(y, p1, p2):
    """Uji DeLong dua sisi untuk AUC berkorelasi. Return (auc1, auc2, z, p)."""
    order = np.argsort(-y, kind='mergesort'); y_s = y[order]
    preds = np.vstack([p1, p2])[:, order]
    m = int(y_s.sum()); n = len(y_s) - m
    tx = np.array([_midrank(r[:m]) for r in preds])
    ty = np.array([_midrank(r[m:]) for r in preds])
    tz = np.array([_midrank(r) for r in preds])
    aucs = tz[:, :m].sum(1) / m / n - (m + 1.0) / 2.0 / n
    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m
    S = np.cov(v01) / m + np.cov(v10) / n
    var = S[0, 0] + S[1, 1] - 2 * S[0, 1]
    if var <= 0:
        return aucs[0], aucs[1], 0.0, 1.0
    z = (aucs[0] - aucs[1]) / np.sqrt(var)
    return aucs[0], aucs[1], float(z), float(2 * stats.norm.sf(abs(z)))

def mcnemar_exact(y, yh1, yh2):
    c1, c2 = (yh1 == y), (yh2 == y)
    b, c = int((c1 & ~c2).sum()), int((~c1 & c2).sum())
    p = 1.0 if b + c == 0 else float(stats.binomtest(min(b, c), b + c, 0.5).pvalue)
    return b, c, p

def holm(pvals):
    pvals = np.asarray(pvals); order = np.argsort(pvals); m = len(pvals)
    adj = np.empty(m); run = 0.0
    for r, i in enumerate(order):
        run = max(run, (m - r) * pvals[i]); adj[i] = min(run, 1.0)
    return adj

pairs = [(FINAL_MODEL, c) for c in CONFIGS if c != FINAL_MODEL]
pairs += [(c + ' +TTA', c) for c in BASE_CFGS]
pairs += [('FPPR', 'Mean'), ('XGB(BB)', 'Mean'), ('XGB(BB+FPPR)', 'FPPR'), ('XGB(BB+FPPR)', 'XGB(BB)')]
seen, uniq = set(), []
for a, b in pairs:
    if a != b and frozenset((a, b)) not in seen:
        seen.add(frozenset((a, b))); uniq.append((a, b))

st_rows = []
for a, b in uniq:
    pa, pb = PROB['test'][a], PROB['test'][b]
    auc_a, auc_b, z, p_dl = delong_test(y_te, pa, pb)
    d_boot = BOOT[a]['AUC'] - BOOT[b]['AUC']
    bb, cc, p_mn = mcnemar_exact(y_te, EVAL[a]['y_pred'], EVAL[b]['y_pred'])
    st_rows.append({'A': a, 'B': b, 'AUC_A': auc_a, 'AUC_B': auc_b,
                    'dAUC': auc_a - auc_b,
                    'dAUC_lo': np.percentile(d_boot, 2.5), 'dAUC_hi': np.percentile(d_boot, 97.5),
                    'p_DeLong': p_dl,
                    'dAcc': EVAL[a]['point']['Accuracy'] - EVAL[b]['point']['Accuracy'],
                    'McNemar_b': bb, 'McNemar_c': cc, 'p_McNemar': p_mn})
df_st = pd.DataFrame(st_rows)
df_st['p_DeLong_Holm']  = holm(df_st['p_DeLong'])
df_st['p_McNemar_Holm'] = holm(df_st['p_McNemar'])

show = df_st.copy()
show['ΔAUC [95% CI]'] = [f'{d:+.4f} [{l:+.4f}, {h:+.4f}]'
                         for d, l, h in zip(show.dAUC, show.dAUC_lo, show.dAUC_hi)]
show['ΔAcc %'] = (show['dAcc'] * 100).map('{:+.2f}'.format)
for col in ['p_DeLong', 'p_DeLong_Holm', 'p_McNemar', 'p_McNemar_Holm']:
    show[col] = show[col].map('{:.4f}'.format)
print(show[['A', 'B', 'ΔAUC [95% CI]', 'p_DeLong', 'p_DeLong_Holm',
            'ΔAcc %', 'p_McNemar', 'p_McNemar_Holm']].to_string(index=False))
df_st.to_csv(os.path.join(OUTPUT, 'results', 'statistical_tests.csv'), index=False)
print('\nInterpretasi: perbedaan dianggap bermakna jika p (Holm) < 0.05 '
      'DAN CI ΔAUC tidak memuat 0.')

---
## 🎚️ SEL 19 — Kalibrasi (R1-6, R2-6)

AUC hanya mengukur **diskriminasi** (urutan skor). Kalibrasi — apakah probabilitas sesuai frekuensi kejadian — diukur dengan **Brier score**, **ECE** dan **reliability diagram**. Catatan: `class_weight` dan `scale_pos_weight` sengaja menggeser probabilitas, sehingga model bisa berdiskriminasi baik tetapi kurang terkalibrasi.

In [ ]:
CAL_SHOW = BASE_CFGS + ([FINAL_MODEL] if FINAL_MODEL not in BASE_CFGS else [])
ncol = 4; nrow = int(np.ceil(len(CAL_SHOW) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.5 * ncol, 4.3 * nrow))
axes = np.atleast_2d(axes)
for ax, c in zip(axes.ravel(), CAL_SHOW):
    p = PROB['test'][c]
    frac, mean_p = calibration_curve(y_te, p, n_bins=ECE_BINS, strategy='uniform')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.6, label='Kalibrasi sempurna')
    ax.plot(mean_p, frac, 'o-', color=PAL['tta'] if c == FINAL_MODEL else PAL['eff'], lw=2,
            label='Model')
    ax2 = ax.twinx(); ax2.hist(p, bins=ECE_BINS, range=(0, 1), color='grey', alpha=0.2)
    ax2.set_yticks([])
    e = EVAL[c]['point']
    ax.set_title(f"{c}{' ★' if c == FINAL_MODEL else ''}\nBrier={e['Brier']:.3f}  ECE={e['ECE']:.3f}",
                 fontsize=9, fontweight='bold')
    ax.set_xlabel('Probabilitas prediksi'); ax.set_ylabel('Fraksi positif aktual')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend(fontsize=7, loc='upper left')
for ax in axes.ravel()[len(CAL_SHOW):]:
    ax.axis('off')
fig.suptitle('Reliability Diagram — Partisi Test', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT, 'figures', 'calibration.png'), dpi=300, bbox_inches='tight')
plt.show()
print(df_num[['Model', 'Brier', 'Brier_lo', 'Brier_hi', 'ECE', 'ECE_lo', 'ECE_hi']]
      .round(4).to_string(index=False))

---
## 📉 SEL 20 — Confusion Matrix & ROC Curve (Test)

In [ ]:
CFG_COL = {'EfficientNetB4': PAL['eff'], 'DenseNet121': PAL['den'], 'ResNet50V2': PAL['res'],
           'Mean': PAL['mean'], 'FPPR': PAL['fppr'], 'XGB(BB)': PAL['xgb'], 'XGB(BB+FPPR)': PAL['xgbf']}
col_of = lambda c: CFG_COL[c.replace(' +TTA', '')]

CM_SHOW = BASE_CFGS + ([FINAL_MODEL] if FINAL_MODEL not in BASE_CFGS else [])
fig, axes = plt.subplots(2, 4, figsize=(20, 9))
for ax, c in zip(axes.ravel(), CM_SHOW):
    cm   = confusion_matrix(y_te, EVAL[c]['y_pred'])
    cm_n = cm.astype(float) / cm.sum(axis=1)[:, None] * 100
    ann  = np.array([[f'{v}\n({cm_n[i, j]:.1f}%)' for j, v in enumerate(row)] for i, row in enumerate(cm)])
    sns.heatmap(cm, annot=ann, fmt='', cmap='Reds' if c == FINAL_MODEL else 'Blues', ax=ax,
                xticklabels=['Non-Card', 'Card'], yticklabels=['Non-Card', 'Card'],
                linewidths=1, linecolor='white', cbar=False)
    ax.set_title(f"{c}{' ★' if c == FINAL_MODEL else ''}\nAcc {EVAL[c]['point']['Accuracy']*100:.1f}%",
                 fontweight='bold', fontsize=10)
    ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')
for ax in axes.ravel()[len(CM_SHOW):]:
    ax.axis('off')
fig.suptitle('Confusion Matrix — Partisi Test (threshold 0.5)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT, 'figures', 'confusion_matrices.png'), dpi=300, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, title, cfgs in [(axes[0], 'Tanpa TTA', BASE_CFGS),
                        (axes[1], 'Dengan TTA', [c + ' +TTA' for c in BASE_CFGS])]:
    for c in cfgs:
        fpr, tpr, _ = roc_curve(y_te, PROB['test'][c])
        lo, hi = EVAL[c]['ci']['AUC']
        ax.plot(fpr, tpr, color=col_of(c), lw=3 if c == FINAL_MODEL else 1.8,
                label=f"{c}{' ★' if c == FINAL_MODEL else ''}  AUC={EVAL[c]['point']['AUC']:.3f} [{lo:.3f}–{hi:.3f}]")
    ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
    ax.set_title(f'ROC — {title}', fontweight='bold')
    ax.set_xlabel('False Positive Rate (1 − spesifisitas)'); ax.set_ylabel('True Positive Rate (sensitivitas)')
    ax.legend(fontsize=8, loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT, 'figures', 'roc_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

---
## 🧩 SEL 21 — SHAP pada XGBoost Meta-Learner

SHAP menjelaskan **kontribusi fitur meta** (probabilitas backbone) terhadap keputusan XGBoost — bukan bukti kausal anatomi (R2-7). Catatan: fitur `-Non` = 1 − `-Card`, dan FPPR adalah kombinasi linear ketiga backbone, sehingga hanya 3 dari 8 dimensi yang independen.

In [ ]:
SHAP_CFG = 'XGB(BB+FPPR)'
xgb_meta = XGB_MODELS[SHAP_CFG]
Xm_shap, Xt_shap = META_X[SHAP_CFG]

explainer = shap.TreeExplainer(xgb_meta)
sv = explainer.shap_values(Xt_shap)
if isinstance(sv, list): sv = sv[1]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'SHAP — {SHAP_CFG} (dilatih di partisi meta, dijelaskan di test)',
             fontsize=12, fontweight='bold')
plt.sca(axes[0])
shap.summary_plot(sv, Xt_shap, feature_names=FEAT_BBF, show=False, max_display=8, plot_size=None)
axes[0].set_title('SHAP Beeswarm — kelas Cardiomegaly', fontweight='bold')

ms = np.abs(sv).mean(0); si = np.argsort(ms)[::-1]
cmap_f = [PAL['eff']]*2 + [PAL['den']]*2 + [PAL['res']]*2 + [PAL['fppr']]*2
axes[1].barh([FEAT_BBF[i] for i in si], ms[si], color=[cmap_f[i] for i in si], alpha=0.85)
axes[1].invert_yaxis()
axes[1].set_title('Mean |SHAP| — Feature Importance', fontweight='bold')
axes[1].set_xlabel('Mean |SHAP value|'); axes[1].grid(axis='x', alpha=0.3)
from matplotlib.patches import Patch
axes[1].legend(handles=[Patch(facecolor=PAL[k], label=l) for k, l in
                        [('eff','EfficientNetB4'), ('den','DenseNet121'), ('res','ResNet50V2'), ('fppr','FPPR')]],
               fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT, 'figures', 'shap.png'), dpi=300, bbox_inches='tight')
plt.show()

---
## 📋 SEL 22 — Tabel Perbandingan Literatur (Bukan Head-to-Head)

Nilai pembanding diambil dari publikasi masing-masing dengan **dataset, subset, dan protokol berbeda**, sehingga **tidak dapat** dipakai untuk mengklaim superioritas (R1-8). Pembanding yang setara (protokol identik) adalah konfigurasi ablation di SEL 17. ⚠️ Verifikasi setiap baris terhadap sumber aslinya (R1-9) sebelum dipakai di manuskrip.

In [ ]:
f = FINAL_MODEL; e = EVAL[f]
lit = [
    ('EfficientNetB7',  'Kaggle Cardiomegaly', '94.0%', '-', 'Academia.edu 2024 — verifikasi'),
    ('DenseNet121',     'NIH (718 citra)',     '95.2%', '-', 'MDPI App. Sci. 2024 — verifikasi'),
    ('CNN AdaMax',      'NIH 112K',            '99.9%', '-', 'Sorour IJIS 2024 — verifikasi'),
    ('Cardio-XAttNet',  'ChestX-ray14',        '88.0%', '0.89', 'Sci. Rep. 2023 — verifikasi'),
    (f'CELM-Cardio ({f})', 'NIH, split per pasien (ini)',
     f"{e['point']['Accuracy']*100:.2f}% [{e['ci']['Accuracy'][0]*100:.1f}–{e['ci']['Accuracy'][1]*100:.1f}]",
     f"{e['point']['AUC']:.3f} [{e['ci']['AUC'][0]:.3f}–{e['ci']['AUC'][1]:.3f}]",
     'Penelitian ini'),
]
df_lit = pd.DataFrame(lit, columns=['Metode', 'Dataset / protokol', 'Accuracy', 'AUC', 'Sumber'])
df_lit['Catatan'] = ['Protokol berbeda — bukan perbandingan langsung'] * 4 + ['Test tunggal, CI bootstrap per pasien']
print(df_lit.to_string(index=False))
df_lit.to_csv(os.path.join(OUTPUT, 'results', 'literature_comparison.csv'), index=False)

---
## 🎯 SEL 23 — Dashboard Ringkasan

In [ ]:
fig = plt.figure(figsize=(22, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.30)
fig.suptitle(f'CELM-Cardio v7 — Ringkasan (AUG_MODE={AUG_MODE}) · Model final dipilih di partisi meta: {FINAL_MODEL}',
             fontsize=13, fontweight='bold')

# [0,0] AUC ± CI semua konfigurasi
ax = fig.add_subplot(gs[0, 0])
order = sorted(CONFIGS, key=lambda c: EVAL[c]['point']['AUC'])
vals  = [EVAL[c]['point']['AUC'] for c in order]
err   = np.array([[v - EVAL[c]['ci']['AUC'][0], EVAL[c]['ci']['AUC'][1] - v] for c, v in zip(order, vals)]).T
bars = ax.barh(order, vals, xerr=err, color=[col_of(c) for c in order], alpha=0.85, capsize=3)
for b_, c in zip(bars, order):
    if c == FINAL_MODEL:
        b_.set_edgecolor('black'); b_.set_linewidth(2.5)
ax.set_yticks(range(len(order)))
ax.set_yticklabels([c + (' ★' if c == FINAL_MODEL else '') for c in order], fontsize=8)
ax.set_xlim(max(0.5, min(EVAL[c]['ci']['AUC'][0] for c in CONFIGS) - 0.03), 1.0)
ax.set_title('AUC test [95% CI bootstrap per pasien]', fontweight='bold')
ax.grid(axis='x', alpha=0.3)

# [0,1] ROC final + backbone
ax = fig.add_subplot(gs[0, 1])
for c in dict.fromkeys(['EfficientNetB4', 'DenseNet121', 'ResNet50V2', FINAL_MODEL]):
    fpr, tpr, _ = roc_curve(y_te, PROB['test'][c])
    ax.plot(fpr, tpr, color=col_of(c), lw=3 if c == FINAL_MODEL else 1.5,
            label=f"{c}{' ★' if c == FINAL_MODEL else ''} ({EVAL[c]['point']['AUC']:.3f})")
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_title('ROC — backbone vs model final', fontweight='bold')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# [0,2] CM final
ax = fig.add_subplot(gs[0, 2])
cm = confusion_matrix(y_te, EVAL[FINAL_MODEL]['y_pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=ax, cbar=False,
            xticklabels=['Non-Card', 'Card'], yticklabels=['Non-Card', 'Card'])
ax.set_title(f'Confusion matrix — {FINAL_MODEL}', fontweight='bold')
ax.set_xlabel('Prediksi'); ax.set_ylabel('Aktual')

# [1,0] reliability final
ax = fig.add_subplot(gs[1, 0])
frac, mean_p = calibration_curve(y_te, PROB['test'][FINAL_MODEL], n_bins=ECE_BINS)
ax.plot([0, 1], [0, 1], 'k--', lw=1); ax.plot(mean_p, frac, 'o-', color=PAL['tta'], lw=2)
ax.set_title(f"Kalibrasi — Brier {EVAL[FINAL_MODEL]['point']['Brier']:.3f} · ECE {EVAL[FINAL_MODEL]['point']['ECE']:.3f}",
             fontweight='bold')
ax.set_xlabel('Probabilitas prediksi'); ax.set_ylabel('Fraksi positif'); ax.grid(alpha=0.3)

# [1,1] val accuracy curves
ax = fig.add_subplot(gs[1, 1])
for hist, ck, colr, lbl in [(hist_eff, ckpt_eff, PAL['eff'], 'EfficientNetB4'),
                            (hist_den, ckpt_den, PAL['den'], 'DenseNet121'),
                            (hist_res, ckpt_res, PAL['res'], 'ResNet50V2')]:
    ax.plot(range(1, len(hist['val_accuracy']) + 1), hist['val_accuracy'], color=colr, lw=2, label=lbl)
    ax.scatter([ck['epoch']], [ck['val_accuracy_history']], color=colr, s=60, zorder=5)
ax.set_title('Val accuracy (titik = checkpoint)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# [1,2] ringkasan teks
ax = fig.add_subplot(gs[1, 2]); ax.axis('off')
e = EVAL[FINAL_MODEL]
info = [
    ('Partisi (citra/pasien)', ' · '.join(f"{s} {SPLIT_COUNTS[s]['images']}/{SPLIT_COUNTS[s]['patients']}" for s in SPLITS)),
    ('Bobot FPPR', f'eff {W_EFF:.4f} · den {W_DEN:.4f} · res {W_RES:.4f}'),
    ('Model final', FINAL_MODEL),
    ('AUC meta (seleksi)', f'{META_AUC[FINAL_MODEL]:.4f}'),
    ('AUC test', f"{e['point']['AUC']:.4f} [{e['ci']['AUC'][0]:.4f}–{e['ci']['AUC'][1]:.4f}]"),
    ('Accuracy test', f"{e['point']['Accuracy']*100:.2f}% [{e['ci']['Accuracy'][0]*100:.2f}–{e['ci']['Accuracy'][1]*100:.2f}]"),
    ('Sens / Spec @0.5', f"{e['point']['Sensitivity']*100:.1f}% / {e['point']['Specificity']*100:.1f}%"),
    (f"Sens / Spec @{e['thr']:.2f}", f"{e['op']['Sensitivity']*100:.1f}% / {e['op']['Specificity']*100:.1f}%"),
    ('Brier / ECE', f"{e['point']['Brier']:.3f} / {e['point']['ECE']:.3f}"),
]
for i, (k, v) in enumerate(info):
    ax.text(0.0, 0.95 - i * 0.105, k + ':', transform=ax.transAxes, fontsize=9, fontweight='bold', color='#444')
    ax.text(0.38, 0.95 - i * 0.105, v, transform=ax.transAxes, fontsize=9, color='#1565C0')
ax.set_title('Ringkasan', fontweight='bold')

plt.savefig(os.path.join(OUTPUT, 'figures', 'dashboard.png'), dpi=200, bbox_inches='tight')
plt.show()

---
## 🌍 SEL 24 — Uji Eksternal CheXpert (R1-7, R2-2)

Model **tidak dilatih ulang**. Bobot FPPR, model XGBoost, model final, dan threshold Youden semuanya **dikunci** dari partisi NIH (val/meta). CheXpert berasal dari institusi lain (Stanford Hospital), labeler lain (CheXpert labeler), dan proporsi foto AP (portable) yang jauh lebih besar → menguji pergeseran domain akuisisi, populasi, dan kebijakan label.

| Skenario | Positif | Negatif | Tujuan |
|---|---|---|---|
| **A — setara** | Cardiomegaly = 1, tanpa temuan lain | No Finding = 1 | Definisi tugas identik dengan NIH → efek domain murni |
| **B — realistis** | Cardiomegaly = 1 (**termasuk komorbid**) | Cardiomegaly = 0 (normal **maupun kelainan lain**), Enlarged Cardiomediastinum ≠ 1/−1 | Heterogenitas klinis yang diminta reviewer |

Hanya foto **frontal**; label tidak pasti (−1) pada Cardiomegaly dikeluarkan; **1 citra per pasien** (studi pertama) sehingga tidak ada korelasi antar-citra. Catatan: label CheXpert set train berasal dari labeler otomatis atas laporan radiologi (bukan anotasi per-citra).

In [ ]:
# ── Unduh CheXpert-v1.0-small (Kaggle: ashery/chexpert, ±11 GB) ───────────────
# Jika sudah ada di Drive, isi CHEXPERT_DIR ke folder yang berisi 'CheXpert-v1.0-small'.
CHEXPERT_SLUG = 'ashery/chexpert'
CHEXPERT_DIR  = '/content/chexpert'
os.makedirs(CHEXPERT_DIR, exist_ok=True)
_need_chx = not glob.glob(os.path.join(CHEXPERT_DIR, '**', 'train.csv'), recursive=True)
print('📥 Download CheXpert ...' if _need_chx else '✅ CheXpert sudah ada — download dilewati')
if _need_chx:
    subprocess.run(['kaggle', 'datasets', 'download', '-d', CHEXPERT_SLUG,
                    '-p', CHEXPERT_DIR, '--unzip', '-q'], check=True)
print(sorted(os.listdir(CHEXPERT_DIR))[:10])

In [ ]:
# ── Bentuk kohort eksternal A & B ─────────────────────────────────────────────
CHX_N_PER_CLASS = 1000      # maks citra per kelas per skenario (1 citra per pasien)
CHX_OTHER = ['Enlarged Cardiomediastinum', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation',
             'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture']
EXT_DIR = '/content/chexpert_eval'

csv_chx  = sorted(glob.glob(os.path.join(CHEXPERT_DIR, '**', 'train.csv'), recursive=True))[0]
chx      = pd.read_csv(csv_chx)
chx_root = os.path.dirname(os.path.dirname(csv_chx))       # folder induk 'CheXpert-v1.0-small'
chx['abs_path'] = chx['Path'].map(lambda p: os.path.join(chx_root, p))
chx = chx[chx['Frontal/Lateral'] == 'Frontal'].copy()
chx['patient'] = chx['Path'].str.extract(r'(patient\d+)')[0]
chx['study']   = chx['Path'].str.extract(r'(study\d+)')[0]
chx['view_f']  = chx['Path'].str.extract(r'(view\d+_\w+)\.jpg')[0]
chx['study_n'] = chx['study'].str.replace('study', '').astype(int)
chx = chx.sort_values(['patient', 'study_n', 'Path']).reset_index(drop=True)
L = lambda col: chx[col].fillna(0)          # kosong = tidak disebut → 0 (konvensi CheXpert)
card, nof, ecm = L('Cardiomegaly'), L('No Finding'), L('Enlarged Cardiomediastinum')
other_pos = (chx[CHX_OTHER].fillna(0) == 1).any(axis=1)

EXT_DEF = {
    'A': {'pos': (card == 1) & ~other_pos, 'neg': (nof == 1) & (card == 0),
          'desc': 'Cardiomegaly tunggal vs No Finding (setara NIH)'},
    'B': {'pos': (card == 1), 'neg': (card == 0) & (ecm == 0),
          'desc': 'Cardiomegaly (termasuk komorbid) vs semua non-kardiomegali (normal & kelainan lain)'},
}

def build_cohort(pos, neg):
    rng_seed = SEED
    p = chx[pos].drop_duplicates('patient')                      # studi pertama per pasien
    p = p.sample(min(CHX_N_PER_CLASS, len(p)), random_state=rng_seed)
    n = chx[neg & ~chx['patient'].isin(p['patient'])].drop_duplicates('patient')
    n = n.sample(min(CHX_N_PER_CLASS, len(n)), random_state=rng_seed)
    d = pd.concat([p.assign(cls='Cardiomegaly'), n.assign(cls='Non_Cardiomegaly')], ignore_index=True)
    d = d[d['abs_path'].map(os.path.exists)].reset_index(drop=True)
    d['fname'] = d['patient'] + '_' + d['study'] + '_' + d['view_f'] + '.jpg'
    assert d['patient'].is_unique
    return d

if os.path.exists(EXT_DIR):
    shutil.rmtree(EXT_DIR)
EXT_COHORTS, EXT_INFO, EXT_F2PID, EXT_F2VIEW = {}, {}, {}, {}
for scen, dfn in EXT_DEF.items():
    d = build_cohort(dfn['pos'], dfn['neg'])
    for cls in CLASSES:
        os.makedirs(os.path.join(EXT_DIR, scen, cls), exist_ok=True)
    for src_, cls, fn in zip(d['abs_path'], d['cls'], d['fname']):
        shutil.copy(src_, os.path.join(EXT_DIR, scen, cls, fn))
    EXT_COHORTS[scen] = d
    EXT_F2PID.update(dict(zip(d['fname'], d['patient'])))
    EXT_F2VIEW.update(dict(zip(d['fname'], d['AP/PA'].fillna('NA'))))
    n_other = int(((d['cls'] == 'Non_Cardiomegaly') & (d['No Finding'].fillna(0) != 1)).sum())
    n_comorb = int(((d['cls'] == 'Cardiomegaly') & (d[CHX_OTHER].fillna(0) == 1).any(axis=1)).sum())
    EXT_INFO[scen] = {'definition': dfn['desc'], 'n_images': int(len(d)), 'n_patients': int(d['patient'].nunique()),
                      'n_cardiomegaly': int((d['cls'] == 'Cardiomegaly').sum()),
                      'n_non_cardiomegaly': int((d['cls'] == 'Non_Cardiomegaly').sum()),
                      'cardiomegaly_with_comorbidity': n_comorb,
                      'negatives_with_other_abnormality': n_other,
                      'view_counts': d['AP/PA'].fillna('NA').value_counts().to_dict()}
    print(f"\n🌍 Skenario {scen}: {dfn['desc']}")
    print(f"   {EXT_INFO[scen]['n_cardiomegaly']} Cardiomegaly · {EXT_INFO[scen]['n_non_cardiomegaly']} Non  "
          f"({EXT_INFO[scen]['n_patients']} pasien, 1 citra/pasien)")
    print(f"   Cardiomegaly dengan komorbid: {n_comorb} · Negatif dengan kelainan lain: {n_other}")
    print(f"   Proyeksi: {EXT_INFO[scen]['view_counts']}")

In [ ]:
# ── Prediksi (tanpa retraining) & evaluasi dengan threshold terkunci ──────────
def configs_from_backbone_probs(Pk):
    """Pk[key][mode] → {config: prob Cardiomegaly}; memakai bobot FPPR & XGBoost dari NIH."""
    out = {}
    for mode, sfx in [('plain', ''), ('tta', ' +TTA')]:
        Pe, Pd, Pr = (Pk[k][mode] for k in BB_KEYS)
        for n_, p_ in simple_ensembles(Pe, Pd, Pr).items():
            out[n_ + sfx] = p_
        for xname, wf in [('XGB(BB)', False), ('XGB(BB+FPPR)', True)]:
            out[xname + sfx] = XGB_MODELS[xname + sfx].predict_proba(meta_features(Pe, Pd, Pr, wf))[:, 1]
    return out

def evaluate_set(y, probs, groups, label):
    """Metrik @0.5, @threshold terkunci, dan CI bootstrap per pasien untuk semua konfigurasi."""
    bidx = make_boot_idx(y, groups)
    res = {}
    for c in CONFIGS:
        p = probs[c]
        bs = np.array([[point_metrics(y[i], p[i])[m] for m in METRICS] for i in bidx])
        res[c] = {'point': point_metrics(y, p), 'op': point_metrics(y, p, THR[c]),
                  'ci': {m: (float(np.percentile(bs[:, j], 2.5)), float(np.percentile(bs[:, j], 97.5)))
                         for j, m in enumerate(METRICS)},
                  'boot_auc': bs[:, 0], 'y_pred': (p >= 0.5).astype(int)}
    print(f'   {label}: {len(y)} citra, bootstrap {len(bidx)} resample')
    return res

EXT_P, EXT_FILES, EXT_Y, EXT_PROB, EXT_EVAL, EXT_PID = {}, {}, {}, {}, {}, {}
for scen in EXT_COHORTS:
    d = os.path.join(EXT_DIR, scen)
    EXT_P[scen] = {}
    for name, k, _, pre in BACKBONES:
        g = eval_gen(pre, d)
        EXT_P[scen][k] = {'plain': MODELS[k].predict(g, verbose=0), 'tta': tta_predict(MODELS[k], pre, d)}
        if scen not in EXT_FILES:
            EXT_FILES[scen], EXT_Y[scen] = list(g.filenames), np.asarray(g.classes)
        else:
            assert EXT_FILES[scen] == list(g.filenames)
    EXT_PID[scen]  = np.array([EXT_F2PID[os.path.basename(f)] for f in EXT_FILES[scen]])
    EXT_PROB[scen] = configs_from_backbone_probs(EXT_P[scen])
    EXT_EVAL[scen] = evaluate_set(EXT_Y[scen], EXT_PROB[scen], EXT_PID[scen], f'CheXpert-{scen}')

# ── Tabel: internal vs eksternal ──────────────────────────────────────────────
def _auc_ci(e):
    return f"{e['point']['AUC']:.3f} [{e['ci']['AUC'][0]:.3f}–{e['ci']['AUC'][1]:.3f}]"

rows = []
for c in CONFIGS:
    r_ = {'Model': c + (' ★' if c == FINAL_MODEL else ''), 'AUC NIH-test': _auc_ci(EVAL[c])}
    for scen in EXT_EVAL:
        e = EXT_EVAL[scen][c]
        r_[f'AUC CheXpert-{scen}'] = _auc_ci(e)
        r_[f'Sens/Spec@thr {scen} %'] = f"{e['op']['Sensitivity']*100:.1f} / {e['op']['Specificity']*100:.1f}"
    r_['ΔAUC (B − NIH)'] = f"{EXT_EVAL['B'][c]['point']['AUC'] - EVAL[c]['point']['AUC']:+.3f}"
    rows.append(r_)
df_ext = pd.DataFrame(rows)
print('\n' + '═' * 130)
print('🌍 VALIDASI EKSTERNAL — model & threshold terkunci dari NIH (tanpa retraining)')
print('═' * 130)
print(df_ext.to_string(index=False))

num = []
for scen in EXT_EVAL:
    for c in CONFIGS:
        e = EXT_EVAL[scen][c]; r_ = {'Scenario': scen, 'Model': c, 'Final': c == FINAL_MODEL}
        for m in METRICS:
            r_[m] = e['point'][m]; r_[m + '_lo'], r_[m + '_hi'] = e['ci'][m]
        r_.update({'Threshold_meta': THR[c], 'Sensitivity_at_thr': e['op']['Sensitivity'],
                   'Specificity_at_thr': e['op']['Specificity']})
        num.append(r_)
pd.DataFrame(num).to_csv(os.path.join(OUTPUT, 'results', 'external_chexpert_metrics.csv'), index=False)

# ── Uji berpasangan di data eksternal: model final vs backbone & Level 1 ──────
st = []
for scen in EXT_EVAL:
    y_, P_ = EXT_Y[scen], EXT_PROB[scen]
    for b in ['EfficientNetB4', 'DenseNet121', 'ResNet50V2', 'Mean', 'FPPR']:
        if b == FINAL_MODEL: continue
        a1, a2, z, p_ = delong_test(y_, P_[FINAL_MODEL], P_[b])
        d_ = EXT_EVAL[scen][FINAL_MODEL]['boot_auc'] - EXT_EVAL[scen][b]['boot_auc']
        st.append({'Scenario': scen, 'A': FINAL_MODEL, 'B': b, 'dAUC': a1 - a2,
                   'dAUC_lo': np.percentile(d_, 2.5), 'dAUC_hi': np.percentile(d_, 97.5), 'p_DeLong': p_})
df_ext_st = pd.DataFrame(st); df_ext_st['p_Holm'] = holm(df_ext_st['p_DeLong'])
print('\n📐 Model final vs pembanding (eksternal):')
print(df_ext_st.round(4).to_string(index=False))
df_ext_st.to_csv(os.path.join(OUTPUT, 'results', 'external_statistical_tests.csv'), index=False)

# ── Subgrup proyeksi AP vs PA (skenario B, model final) ──────────────────────
print(f'\n🩻 Subgrup proyeksi — CheXpert-B, {FINAL_MODEL}:')
views = np.array([EXT_F2VIEW[os.path.basename(f)] for f in EXT_FILES['B']])
EXT_INFO['B']['by_view'] = {}
for v in ['PA', 'AP']:
    m = views == v
    if m.sum() > 20 and len(np.unique(EXT_Y['B'][m])) == 2:
        yv, pv = EXT_Y['B'][m], EXT_PROB['B'][FINAL_MODEL][m]
        bi = make_boot_idx(yv, EXT_PID['B'][m])
        aucs = [roc_auc_score(yv[i], pv[i]) for i in bi]
        a = roc_auc_score(yv, pv); lo, hi = np.percentile(aucs, [2.5, 97.5])
        op = point_metrics(yv, pv, THR[FINAL_MODEL])
        EXT_INFO['B']['by_view'][v] = {'n': int(m.sum()), 'AUC': float(a), 'AUC_CI': [float(lo), float(hi)],
                                      'sens_at_thr': op['Sensitivity'], 'spec_at_thr': op['Specificity']}
        print(f"   {v}: n={m.sum():4d}  AUC={a:.3f} [{lo:.3f}–{hi:.3f}]  "
              f"Sens/Spec@thr={op['Sensitivity']*100:.1f}/{op['Specificity']*100:.1f}%")
    else:
        print(f'   {v}: sampel terlalu sedikit')

# ── Simpan prediksi per sampel ────────────────────────────────────────────────
for scen in EXT_EVAL:
    d = pd.DataFrame({'file': EXT_FILES[scen], 'patient': EXT_PID[scen], 'y_true': EXT_Y[scen],
                      'view': [EXT_F2VIEW[os.path.basename(f)] for f in EXT_FILES[scen]]})
    for c in CONFIGS:
        d['p_' + c.replace(' +TTA', '_tta').replace('(', '_').replace(')', '').replace('+', '_')] = EXT_PROB[scen][c]
    d.to_csv(os.path.join(OUTPUT, 'results', f'predictions_chexpert_{scen}.csv'), index=False)
json.dump(EXT_INFO, open(os.path.join(OUTPUT, 'results', 'external_chexpert_info.json'), 'w'), indent=2, default=str)

# ── Gambar: ROC model final & AUC per konfigurasi ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
ax = axes[0]
for lbl, y_, p_, col in [('NIH test (internal)', y_te, PROB['test'][FINAL_MODEL], PAL['eff']),
                         ('CheXpert-A (setara)', EXT_Y['A'], EXT_PROB['A'][FINAL_MODEL], PAL['fppr']),
                         ('CheXpert-B (realistis)', EXT_Y['B'], EXT_PROB['B'][FINAL_MODEL], PAL['tta'])]:
    fpr, tpr, _ = roc_curve(y_, p_)
    ax.plot(fpr, tpr, color=col, lw=2.5, label=f'{lbl}  AUC={roc_auc_score(y_, p_):.3f}')
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_title(f'ROC — {FINAL_MODEL}: internal vs eksternal', fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
xs_ = np.arange(len(CONFIGS)); wd = 0.27
for j, (lbl, getter, col) in enumerate([('NIH test', lambda c: EVAL[c], PAL['eff']),
                                        ('CheXpert-A', lambda c: EXT_EVAL['A'][c], PAL['fppr']),
                                        ('CheXpert-B', lambda c: EXT_EVAL['B'][c], PAL['tta'])]):
    v = np.array([getter(c)['point']['AUC'] for c in CONFIGS])
    lo = np.array([getter(c)['ci']['AUC'][0] for c in CONFIGS]); hi = np.array([getter(c)['ci']['AUC'][1] for c in CONFIGS])
    ax.bar(xs_ + (j - 1) * wd, v, wd, yerr=[v - lo, hi - v], color=col, alpha=0.85, capsize=2, label=lbl)
ax.set_xticks(xs_); ax.set_xticklabels([c + (' ★' if c == FINAL_MODEL else '') for c in CONFIGS], rotation=60, ha='right', fontsize=8)
ax.set_ylim(0.4, 1.0); ax.set_ylabel('AUC [95% CI]'); ax.legend(); ax.grid(axis='y', alpha=0.3)
ax.set_title('AUC per konfigurasi — internal vs eksternal', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT, 'figures', 'external_chexpert.png'), dpi=300, bbox_inches='tight')
plt.show()

---
## 📏 SEL 25 — Pengukuran CTR Otomatis & Analisis Galat (R1-10, R2-7, R1-8)

**Prosedur (terdokumentasi, dapat direproduksi):**
1. **Segmentasi** jantung, paru kiri, dan paru kanan dengan model publik `torchxrayvision` PSPNet (`chestx_det`, dilatih pada ChestX-Det; 14 struktur anatomi; input 512×512, center-crop). Opsi alternatif: detektor YOLO ASD-CXR milik sendiri (`CTR_METHOD = 'yolo'`).
2. **Pembersihan mask**: komponen terhubung terbesar + *morphological opening* 15 px.
3. **MHD** (maximal horizontal cardiac diameter) = lebar horizontal maksimum siluet jantung.
4. **MTD** (maximal internal thoracic diameter) = jarak horizontal terjauh antara tepi lateral paru kanan dan kiri (baris terlebar).
5. **CTR = MHD / MTD**; kardiomegali jika **CTR > 0.50** (konvensi PA).
6. **QC otomatis**: mask jantung & kedua paru harus ada, CTR dalam rentang 0.25–0.85, pusat jantung di antara paru. Citra gagal QC dilaporkan, tidak diimputasi.

**Implementasi:** segmentasi dijalankan di proses Python terpisah (`ctr_worker.py`, hanya PyTorch) agar tidak bentrok dengan TensorFlow; skrip worker tersimpan di folder `ctr/` untuk reprodusibilitas.

**Batasan:** CTR otomatis bukan *ground truth* radiolog, dan CTR pada proyeksi AP cenderung lebih besar. Karena itu sel terakhir mengekspor 50 citra (overlay + template CSV) untuk **verifikasi manual** dan menghitung kesepakatan (ICC, Bland–Altman) bila template sudah diisi.

In [ ]:
# ── Konfigurasi CTR + worker terpisah ─────────────────────────────────────────
# Segmentasi (PyTorch) dijalankan di proses Python terpisah: TensorFlow & PyTorch dalam satu
# proses dapat bentrok (segfault alokator/OpenMP), dan cara ini juga membebaskan memori GPU.
CTR_METHOD   = 'xrv'      # 'xrv' (torchxrayvision PSPNet, publik) | 'yolo' (detektor ASD-CXR sendiri)
CTR_CUTOFF   = 0.50       # kardiomegali jika CTR > 0.50
YOLO_WEIGHTS = '/content/drive/MyDrive/ASD-CXR/best.pt'   # hanya untuk CTR_METHOD='yolo'
YOLO_HEART, YOLO_LUNG = 'heart', 'lungs'                  # nama kelas pada model YOLO

CTR_DIR = os.path.join(OUTPUT, 'ctr'); os.makedirs(CTR_DIR, exist_ok=True)
CTR_WORKER = os.path.join(CTR_DIR, 'ctr_worker.py')
with open(CTR_WORKER, 'w') as fh:
    fh.write('"""ctr_worker.py — pengukuran CTR di proses terpisah (hanya PyTorch, tanpa TensorFlow).\nPemakaian: python ctr_worker.py task.json\ntask = {"mode": "measure"|"overlay", "method": "xrv"|"yolo", "items": [...], ...}"""\nimport sys, json, os\nimport numpy as np, cv2, pandas as pd\n\nT = json.load(open(sys.argv[1]))\nMETHOD = T[\'method\']\n\nif METHOD == \'xrv\':\n    import torch, torchvision\n    import torchxrayvision as xrv\n    from scipy import ndimage\n    SEG = xrv.baseline_models.chestx_det.PSPNet().eval()\n    DEV = \'cuda\' if torch.cuda.is_available() else \'cpu\'\n    SEG = SEG.to(DEV)\n    TF_ = torchvision.transforms.Compose([xrv.datasets.XRayCenterCrop(), xrv.datasets.XRayResizer(512)])\n\n    def largest_cc(mask):\n        lab, n = ndimage.label(mask)\n        if n == 0:\n            return mask\n        sizes = ndimage.sum(mask, lab, range(1, n + 1))\n        return lab == (1 + int(np.argmax(sizes)))\n\n    def xrv_image(path):\n        g = cv2.imread(path, cv2.IMREAD_GRAYSCALE).astype(np.float32)\n        return TF_(xrv.datasets.normalize(g, 255)[None, ...])\n\n    def segment(path):\n        x = torch.from_numpy(xrv_image(path))[None, ...].to(DEV)\n        with torch.no_grad():\n            p = torch.sigmoid(SEG(x))[0].cpu().numpy()\n        out = {}\n        for k in [\'Heart\', \'Left Lung\', \'Right Lung\']:\n            m = ndimage.binary_opening(p[SEG.targets.index(k)] > 0.5, structure=np.ones((15, 15)))\n            out[k] = largest_cc(m)\n        return out\n\n    def ctr_from_masks(mk):\n        heart = mk[\'Heart\']; lungs = mk[\'Left Lung\'] | mk[\'Right Lung\']\n        r = {\'ok\': False, \'qc_reason\': \'\'}\n        if heart.sum() < 500:\n            r[\'qc_reason\'] = \'mask jantung kosong/kecil\'; return r\n        if mk[\'Left Lung\'].sum() < 1500 or mk[\'Right Lung\'].sum() < 1500:\n            r[\'qc_reason\'] = \'paru tidak lengkap\'; return r\n        hc = np.where(heart.any(0))[0]\n        MHD = int(hc.max() - hc.min() + 1)\n        rows = np.where(lungs.any(1))[0]\n        MTD, row = max((int(np.ptp(np.where(lungs[rw])[0]) + 1), int(rw)) for rw in rows)\n        lc = np.where(lungs.any(0))[0]\n        ctr = MHD / MTD\n        r.update(MHD=MHD, MTD=MTD, CTR=float(ctr), mtd_row=row,\n                 heart_x0=int(hc.min()), heart_x1=int(hc.max()),\n                 heart_y=int(np.where(heart.any(1))[0].mean()),\n                 mtd_x0=int(np.where(lungs[row])[0].min()), mtd_x1=int(np.where(lungs[row])[0].max()))\n        if not (0.25 <= ctr <= 0.85):\n            r[\'qc_reason\'] = f\'CTR di luar rentang fisiologis ({ctr:.2f})\'; return r\n        if not (lc.min() < hc.mean() < lc.max()):\n            r[\'qc_reason\'] = \'jantung tidak di antara paru\'; return r\n        r[\'ok\'] = True\n        return r\n\n    def measure(path):\n        mk = segment(path)\n        return ctr_from_masks(mk), mk\nelse:\n    from ultralytics import YOLO\n    YM = YOLO(T[\'yolo_weights\'])\n\n    def measure(path):\n        res = YM(path, verbose=False)[0]\n        names = res.names; xyxy = res.boxes.xyxy.cpu().numpy()\n        cls = res.boxes.cls.cpu().numpy().astype(int); conf = res.boxes.conf.cpu().numpy()\n        hb = [(c_, b) for b, k, c_ in zip(xyxy, cls, conf) if names[k] == T[\'yolo_heart\']]\n        lb = [b for b, k, c_ in zip(xyxy, cls, conf) if names[k] == T[\'yolo_lung\'] and c_ > 0.25]\n        if not hb or not lb:\n            return {\'ok\': False, \'qc_reason\': \'box jantung/paru tidak terdeteksi\'}, None\n        h = max(hb, key=lambda t: t[0])[1]\n        x0, x1 = min(b[0] for b in lb), max(b[2] for b in lb)\n        MHD, MTD = float(h[2] - h[0]), float(x1 - x0)\n        ctr = MHD / MTD; ok = 0.25 <= ctr <= 0.85\n        return {\'ok\': ok, \'qc_reason\': \'\' if ok else f\'CTR di luar rentang ({ctr:.2f})\', \'MHD\': MHD, \'MTD\': MTD,\n                \'CTR\': ctr, \'heart_x0\': h[0], \'heart_x1\': h[2], \'heart_y\': (h[1] + h[3]) / 2,\n                \'mtd_x0\': x0, \'mtd_x1\': x1, \'mtd_row\': max(b[3] for b in lb) - 5}, None\n\nKEYS = [\'ok\', \'qc_reason\', \'MHD\', \'MTD\', \'CTR\']\nif T[\'mode\'] == \'measure\':\n    rows = []\n    n = len(T[\'items\'])\n    for i, (sname, fname, path, y) in enumerate(T[\'items\']):\n        try:\n            r, _ = measure(path)\n        except Exception as e:\n            r = {\'ok\': False, \'qc_reason\': f\'error: {e}\'}\n        rows.append({\'set\': sname, \'file\': fname, \'y_true\': int(y), **{k: r.get(k) for k in KEYS}})\n        if (i + 1) % 100 == 0 or i + 1 == n:\n            print(f\'  {i + 1}/{n}\', flush=True)\n    pd.DataFrame(rows).to_csv(T[\'out_csv\'], index=False)\nelse:   # overlay\n    import matplotlib; matplotlib.use(\'Agg\'); import matplotlib.pyplot as plt\n    for path, title, out_png in T[\'items\']:\n        r, mk = measure(path)\n        fig, ax = plt.subplots(figsize=(5, 5.4))\n        if METHOD == \'xrv\':\n            ax.imshow(xrv_image(path)[0], cmap=\'gray\')\n            ov = np.zeros((512, 512, 4))\n            ov[mk[\'Left Lung\'] | mk[\'Right Lung\']] = [0.1, 0.5, 1.0, 0.25]\n            ov[mk[\'Heart\']] = [1.0, 0.1, 0.1, 0.40]\n            ax.imshow(ov)\n        else:\n            ax.imshow(cv2.imread(path, cv2.IMREAD_GRAYSCALE), cmap=\'gray\')\n        if \'MHD\' in r:\n            ax.plot([r[\'heart_x0\'], r[\'heart_x1\']], [r[\'heart_y\']] * 2, color=\'red\', lw=2)\n            ax.plot([r[\'mtd_x0\'], r[\'mtd_x1\']], [r[\'mtd_row\']] * 2, color=\'cyan\', lw=2)\n        ok = \'OK\' if r[\'ok\'] else \'QC gagal: \' + r[\'qc_reason\']\n        ax.set_title(f"{title}\\nCTR={r.get(\'CTR\', float(\'nan\')):.3f} · {ok}", fontsize=8)\n        ax.axis(\'off\')\n        plt.savefig(out_png, dpi=110, bbox_inches=\'tight\'); plt.close(fig)\n    print(f\'  {len(T["items"])} overlay tersimpan\', flush=True)\n')

def run_ctr_worker(task, name):
    """Jalankan worker CTR dan teruskan log-nya ke output sel."""
    task = {'method': CTR_METHOD, 'yolo_weights': YOLO_WEIGHTS,
            'yolo_heart': YOLO_HEART, 'yolo_lung': YOLO_LUNG, **task}
    tpath = os.path.join(CTR_DIR, f'task_{name}.json')
    json.dump(task, open(tpath, 'w'))
    proc = subprocess.Popen([sys.executable, CTR_WORKER, tpath], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in proc.stdout:
        tail.append(line)
        if line.startswith('  '):
            print(line, end='')
    if proc.wait() != 0:
        raise RuntimeError('Worker CTR gagal:\n' + ''.join(tail[-30:]))

print(f'✅ Worker CTR siap ({CTR_METHOD}): {CTR_WORKER}')

In [ ]:
# ── Ukur CTR: NIH meta & test (+ CheXpert-B bila tersedia) ───────────────────
CTR_SETS = {'meta': (META_DIR, FILES['meta'], y_me), 'test': (TEST_DIR, FILES['test'], y_te)}
if 'EXT_FILES' in globals():
    CTR_SETS['chexpert_B'] = (os.path.join(EXT_DIR, 'B'), EXT_FILES['B'], EXT_Y['B'])

items = [[s, f, os.path.join(d, f), int(y)] for s, (d, files, ys) in CTR_SETS.items() for f, y in zip(files, ys)]
ctr_csv = os.path.join(OUTPUT, 'results', 'ctr_measurements.csv')
print(f'📏 Mengukur CTR untuk {len(items)} citra ...')
run_ctr_worker({'mode': 'measure', 'items': items, 'out_csv': ctr_csv}, 'measure')
df_ctr = pd.read_csv(ctr_csv)
df_ctr['ok'] = df_ctr['ok'].astype(bool)

print('\n🧪 Lolos QC per set:')
print(df_ctr.groupby('set')['ok'].agg(['sum', 'count', 'mean']).rename(
    columns={'sum': 'lolos', 'count': 'total', 'mean': 'proporsi'}).round(3).to_string())
fails = df_ctr.loc[~df_ctr['ok'], 'qc_reason'].value_counts()
if len(fails):
    print('\nAlasan gagal QC:'); print(fails.to_string())

In [ ]:
# ── Analisis CTR (NIH test) ───────────────────────────────────────────────────
def iqr_str(x):
    x = np.asarray(x, float)
    return f'{np.median(x):.3f} [{np.percentile(x, 25):.3f}–{np.percentile(x, 75):.3f}]' if len(x) else '—'

CTR_INFO = {'method': CTR_METHOD, 'cutoff': CTR_CUTOFF,
            'definition': 'CTR = MHD/MTD; MHD = max horizontal heart width; MTD = widest horizontal '
                          'distance between lateral lung borders; QC: masks present, 0.25<=CTR<=0.85, heart between lungs',
            'qc_pass_rate': df_ctr.groupby('set')['ok'].mean().to_dict()}

t = df_ctr[(df_ctr['set'] == 'test')].reset_index(drop=True)
assert list(t['file']) == FILES['test']
okm = t['ok'].astype(bool).values
ctr_te = t['CTR'].astype(float).values
p_fin  = PROB['test'][FINAL_MODEL]
yhat   = EVAL[FINAL_MODEL]['y_pred']
group  = np.where(y_te == 1, np.where(yhat == 1, 'TP', 'FN'), np.where(yhat == 1, 'FP', 'TN'))
# Proyeksi NIH (PA/AP) dari CSV label
_view_col = next((c for c in df.columns if 'view' in c.lower()), None)
IMG2VIEW = dict(zip(df[img_col].astype(str), df[_view_col])) if _view_col else {}
view_te = np.array([IMG2VIEW.get(os.path.basename(f), 'NA') for f in FILES['test']])

print(f'Citra test lolos QC: {okm.sum()}/{len(okm)}')
yq, cq, pq, gq, pidq = y_te[okm], ctr_te[okm], p_fin[okm], group[okm], PID['test'][okm]

# 1) Distribusi CTR per kelas
u = stats.mannwhitneyu(cq[yq == 1], cq[yq == 0], alternative='two-sided')
print(f'\n1) CTR per kelas — Cardiomegaly: {iqr_str(cq[yq==1])} · Non: {iqr_str(cq[yq==0])} '
      f'(Mann–Whitney p={u.pvalue:.2e})')
CTR_INFO['by_class'] = {'cardiomegaly': iqr_str(cq[yq == 1]), 'non_cardiomegaly': iqr_str(cq[yq == 0]),
                        'mannwhitney_p': float(u.pvalue)}

# 2) CTR sebagai baseline klinis (split identik → perbandingan head-to-head)
bi = make_boot_idx(yq, pidq)
auc_ctr = roc_auc_score(yq, cq); auc_fin_q = roc_auc_score(yq, pq)
b_ctr = np.array([roc_auc_score(yq[i], cq[i]) for i in bi]); b_fin = np.array([roc_auc_score(yq[i], pq[i]) for i in bi])
rule = point_metrics(yq, cq, CTR_CUTOFF + 1e-9)                   # CTR > 0.50
okmeta = df_ctr[(df_ctr['set'] == 'meta')]['ok'].astype(bool).values
thr_ctr = youden_threshold(y_me[okmeta], df_ctr[df_ctr['set'] == 'meta']['CTR'].astype(float).values[okmeta])
rule_y = point_metrics(yq, cq, thr_ctr)
_, _, z, p_dl = delong_test(yq, pq, cq)
print(f'\n2) Baseline CTR (n={okm.sum()}):')
print(f'   AUC CTR        = {auc_ctr:.3f} [{np.percentile(b_ctr,2.5):.3f}–{np.percentile(b_ctr,97.5):.3f}]')
print(f'   AUC {FINAL_MODEL:14s}= {auc_fin_q:.3f} [{np.percentile(b_fin,2.5):.3f}–{np.percentile(b_fin,97.5):.3f}]')
print(f'   ΔAUC (model − CTR) = {auc_fin_q-auc_ctr:+.3f} '
      f'[{np.percentile(b_fin-b_ctr,2.5):+.3f}, {np.percentile(b_fin-b_ctr,97.5):+.3f}], DeLong p={p_dl:.4f}')
print(f"   Aturan CTR > {CTR_CUTOFF:.2f}: Sens={rule['Sensitivity']*100:.1f}%  Spec={rule['Specificity']*100:.1f}%  Acc={rule['Accuracy']*100:.1f}%")
print(f"   CTR ≥ {thr_ctr:.3f} (Youden, dari partisi meta): Sens={rule_y['Sensitivity']*100:.1f}%  Spec={rule_y['Specificity']*100:.1f}%")
CTR_INFO['baseline'] = {'n': int(okm.sum()), 'auc_ctr': float(auc_ctr),
                        'auc_ctr_ci': [float(np.percentile(b_ctr, 2.5)), float(np.percentile(b_ctr, 97.5))],
                        'auc_final_same_subset': float(auc_fin_q), 'delong_p': float(p_dl),
                        'rule_0.50': {k: rule[k] for k in ['Sensitivity', 'Specificity', 'Accuracy']},
                        'youden_thr_meta': thr_ctr,
                        'rule_youden': {k: rule_y[k] for k in ['Sensitivity', 'Specificity', 'Accuracy']}}

# 3) Analisis galat model final berdasarkan CTR terukur
print(f'\n3) CTR per kelompok prediksi {FINAL_MODEL} (threshold 0.5):')
grp_rows = []
for gname in ['TP', 'FN', 'TN', 'FP']:
    x = cq[gq == gname]
    grp_rows.append({'Kelompok': gname, 'n': len(x), 'CTR median [IQR]': iqr_str(x),
                     f'% CTR>{CTR_CUTOFF:.2f}': f'{(x > CTR_CUTOFF).mean()*100:.1f}' if len(x) else '—',
                     '% CTR 0.50–0.55': f'{((x >= 0.50) & (x < 0.55)).mean()*100:.1f}' if len(x) else '—'})
df_grp = pd.DataFrame(grp_rows); print(df_grp.to_string(index=False))
CTR_INFO['error_groups'] = df_grp.to_dict(orient='records')
if (gq == 'FN').sum() >= 3 and (gq == 'TP').sum() >= 3:
    u2 = stats.mannwhitneyu(cq[gq == 'FN'], cq[gq == 'TP'], alternative='two-sided')
    print(f'   FN vs TP: Mann–Whitney p={u2.pvalue:.3g}')
    CTR_INFO['FN_vs_TP_p'] = float(u2.pvalue)

# 4) Korelasi probabilitas model dengan CTR
rho, p_rho = stats.spearmanr(pq, cq)
print(f'\n4) Spearman ρ (p_model, CTR) = {rho:.3f} (p={p_rho:.2e})')
CTR_INFO['spearman'] = {'rho': float(rho), 'p': float(p_rho)}

# 5) Proyeksi PA vs AP
print('\n5) CTR per proyeksi (median [IQR]):')
CTR_INFO['by_view'] = {}
for v in ['PA', 'AP']:
    m = view_te[okm] == v
    if m.sum():
        CTR_INFO['by_view'][v] = {'n': int(m.sum()), 'cardiomegaly': iqr_str(cq[m & (yq == 1)]),
                                  'non': iqr_str(cq[m & (yq == 0)])}
        print(f"   {v}: n={m.sum():4d}  Card {iqr_str(cq[m & (yq==1)])}  ·  Non {iqr_str(cq[m & (yq==0)])}")

# 6) CheXpert-B (opsional)
if 'chexpert_B' in CTR_SETS:
    e = df_ctr[df_ctr['set'] == 'chexpert_B'].reset_index(drop=True); okb = e['ok'].astype(bool).values
    yb, cb, pb = EXT_Y['B'][okb], e['CTR'].astype(float).values[okb], EXT_PROB['B'][FINAL_MODEL][okb]
    if len(np.unique(yb)) == 2:
        CTR_INFO['chexpert_B'] = {'n': int(okb.sum()), 'auc_ctr': float(roc_auc_score(yb, cb)),
                                  'auc_final_same_subset': float(roc_auc_score(yb, pb)),
                                  'delong_p': float(delong_test(yb, pb, cb)[3])}
        print(f"\n6) CheXpert-B (n={okb.sum()}): AUC CTR={CTR_INFO['chexpert_B']['auc_ctr']:.3f} · "
              f"AUC {FINAL_MODEL}={CTR_INFO['chexpert_B']['auc_final_same_subset']:.3f} · "
              f"DeLong p={CTR_INFO['chexpert_B']['delong_p']:.4f}")

json.dump(CTR_INFO, open(os.path.join(OUTPUT, 'results', 'ctr_analysis.json'), 'w'), indent=2, default=str)

# ── Gambar ────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(24, 5.8))
ax = axes[0]
ax.boxplot([cq[yq == 0], cq[yq == 1]], widths=0.5)
ax.set_xticks([1, 2]); ax.set_xticklabels(['Non-Cardiomegaly', 'Cardiomegaly'])
for j, cl in enumerate([0, 1]):
    xx = np.random.default_rng(SEED).normal(j + 1, 0.05, (yq == cl).sum())
    ax.scatter(xx, cq[yq == cl], s=6, alpha=0.35, color=[PAL['eff'], PAL['den']][j])
ax.axhline(CTR_CUTOFF, color='red', ls='--', lw=1.5, label=f'CTR = {CTR_CUTOFF:.2f}')
ax.set_ylabel('CTR terukur'); ax.set_title('Distribusi CTR per kelas (NIH test)', fontweight='bold'); ax.legend()

ax = axes[1]
order_g = ['TP', 'FN', 'TN', 'FP']
ax.boxplot([cq[gq == g] if (gq == g).any() else [np.nan] for g in order_g], widths=0.5)
ax.set_xticks(range(1, 5)); ax.set_xticklabels([f'{g}\n(n={(gq==g).sum()})' for g in order_g])
ax.axhline(CTR_CUTOFF, color='red', ls='--', lw=1.5)
ax.axhspan(0.50, 0.55, color='orange', alpha=0.12, label='CTR 0.50–0.55')
ax.set_title(f'CTR per kelompok prediksi — {FINAL_MODEL}', fontweight='bold'); ax.legend()

ax = axes[2]
for lbl, sc, col in [(f'CTR otomatis (AUC={auc_ctr:.3f})', cq, PAL['fppr']),
                     (f'{FINAL_MODEL} (AUC={auc_fin_q:.3f})', pq, PAL['tta'])]:
    fpr, tpr, _ = roc_curve(yq, sc); ax.plot(fpr, tpr, color=col, lw=2.5, label=lbl)
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_title('ROC — model vs baseline CTR (subset lolos QC)', fontweight='bold')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[3]
ax.scatter(cq[yq == 0], pq[yq == 0], s=10, alpha=0.5, color=PAL['eff'], label='Non-Cardiomegaly')
ax.scatter(cq[yq == 1], pq[yq == 1], s=10, alpha=0.5, color=PAL['den'], label='Cardiomegaly')
ax.axvline(CTR_CUTOFF, color='red', ls='--', lw=1); ax.axhline(0.5, color='grey', ls=':', lw=1)
ax.set_xlabel('CTR terukur'); ax.set_ylabel(f'p(Cardiomegaly) — {FINAL_MODEL}')
ax.set_title(f'Probabilitas model vs CTR (Spearman ρ={rho:.2f})', fontweight='bold'); ax.legend(fontsize=9); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT, 'figures', 'ctr_analysis.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ── Contoh overlay QC + paket verifikasi manual (50 citra) ───────────────────
rng_ = np.random.default_rng(SEED)
show = []
for gname in ['TP', 'FN', 'TN', 'FP']:
    idx = np.where(group == gname)[0]
    show += [(gname, int(i)) for i in rng_.choice(idx, size=min(2, len(idx)), replace=False)]
ex_dir = os.path.join(CTR_DIR, 'examples'); os.makedirs(ex_dir, exist_ok=True)
ex_items = [[os.path.join(TEST_DIR, FILES['test'][i]),
             f"{g} · {os.path.basename(FILES['test'][i])} · p={p_fin[i]:.2f}",
             os.path.join(ex_dir, f'{k}_{g}.png')] for k, (g, i) in enumerate(show)]
run_ctr_worker({'mode': 'overlay', 'items': ex_items}, 'examples')
fig, axes = plt.subplots(2, 4, figsize=(18, 9.5))
for ax, it in zip(axes.ravel(), ex_items):
    ax.imshow(plt.imread(it[2])); ax.axis('off')
for ax in axes.ravel()[len(ex_items):]:
    ax.axis('off')
fig.suptitle('Contoh segmentasi & pengukuran CTR (merah = MHD, cyan = MTD)', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT, 'figures', 'ctr_examples.png'), dpi=200, bbox_inches='tight')
plt.show()

# Paket verifikasi manual: 50 citra (dahulukan FN & FP); label kelas sengaja tidak dicantumkan
rev_dir = os.path.join(OUTPUT, 'ctr_manual_review'); os.makedirs(rev_dir, exist_ok=True)
pool = list(np.where(group == 'FN')[0]) + list(np.where(group == 'FP')[0])
pool_set = set(pool)
rest = [i for i in rng_.permutation(len(group)) if i not in pool_set]
pick = (pool + rest)[:50]
rev_items = [[os.path.join(TEST_DIR, FILES['test'][i]), os.path.basename(FILES['test'][i]),
              os.path.join(rev_dir, os.path.basename(FILES['test'][i]).rsplit('.', 1)[0] + '_overlay.png')]
             for i in pick]
run_ctr_worker({'mode': 'overlay', 'items': rev_items}, 'review')
auto = t.set_index('file')
pd.DataFrame([{'image': os.path.basename(FILES['test'][i]),
               'auto_CTR': auto.loc[FILES['test'][i], 'CTR'], 'auto_qc_ok': auto.loc[FILES['test'][i], 'ok'],
               'manual_MHD_px': '', 'manual_MTD_px': '', 'manual_CTR': '', 'reviewer': '', 'notes': ''}
              for i in pick]).to_csv(os.path.join(rev_dir, 'ctr_manual_review_template.csv'), index=False)
print(f'📝 Paket verifikasi manual: {rev_dir} ({len(pick)} overlay + template CSV)')

def ctr_agreement(csv_path):
    """Setelah template diisi radiolog: ICC(2,1) & Bland–Altman antara CTR otomatis dan manual."""
    d = pd.read_csv(csv_path).dropna(subset=['auto_CTR', 'manual_CTR'])
    a, m = d['auto_CTR'].astype(float).values, d['manual_CTR'].astype(float).values
    X = np.vstack([a, m]).T; n, k = X.shape
    gm = X.mean(); msr = k * ((X.mean(1) - gm) ** 2).sum() / (n - 1)
    msc = n * ((X.mean(0) - gm) ** 2).sum() / (k - 1)
    sse = ((X - X.mean(1, keepdims=True) - X.mean(0, keepdims=True) + gm) ** 2).sum()
    mse = sse / ((n - 1) * (k - 1))
    icc = (msr - mse) / (msr + (k - 1) * mse + k * (msc - mse) / n)
    diff = a - m; md_, sd = diff.mean(), diff.std(ddof=1)
    print(f'n={n}  ICC(2,1)={icc:.3f}  bias={md_:+.4f}  LoA=[{md_-1.96*sd:+.4f}, {md_+1.96*sd:+.4f}]')
    return {'n': n, 'icc': icc, 'bias': md_, 'loa': (md_ - 1.96 * sd, md_ + 1.96 * sd)}

# Contoh pemakaian setelah template diisi:
# ctr_agreement(os.path.join(OUTPUT, 'ctr_manual_review', 'ctr_manual_review_template.csv'))

---
## 🗺️ SEL 26 — Diagram Protokol Evaluasi Bebas Kebocoran (R2-3)

Diagram ini (PNG 300 dpi + PDF vektor) bisa langsung dipakai sebagai figure di manuskrip. Angka citra/pasien diisi otomatis dari run ini.

In [ ]:
from matplotlib.patches import FancyBboxPatch

def draw_protocol(path_png, path_pdf):
    fig, ax = plt.subplots(figsize=(15, 10.5)); ax.set_xlim(0, 15); ax.set_ylim(-1.9, 8.5); ax.axis('off')

    def box(x, y, w, h, text, fc, fs=9.5, bold=False):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.05,rounding_size=0.12',
                                    fc=fc, ec='#333', lw=1.2))
        ax.text(x + w / 2, y + h / 2, text, ha='center', va='center', fontsize=fs,
                fontweight='bold' if bold else 'normal', wrap=True)

    def arrow(x1, y1, x2, y2, txt=None, color='#333', ls='-'):
        ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle='-|>', color=color, lw=1.4, ls=ls))
        if txt:
            ax.text((x1 + x2) / 2 + 0.05, (y1 + y2) / 2, txt, fontsize=8, color=color, va='center')

    n_all = sum(SPLIT_COUNTS[s]['images'] for s in SPLITS)
    p_all = sum(SPLIT_COUNTS[s]['patients'] for s in SPLITS)
    box(4.0, 7.3, 7.0, 0.9,
        f'NIH ChestX-ray14 — Cardiomegaly (label tunggal) vs No Finding\n'
        f'{n_all} citra · {p_all} pasien', '#ECEFF1', 10, True)
    box(4.0, 6.0, 7.0, 0.8, 'Split per PASIEN (stratified, seed tetap) → 4 partisi tanpa overlap pasien',
        '#FFF8E1', 9.5, True)
    arrow(7.5, 7.3, 7.5, 6.8)

    xs = [0.3, 4.0, 7.7, 11.4]; w = 3.3
    colors = {'train': '#E3F2FD', 'val': '#E8F5E9', 'meta': '#FFF3E0', 'test': '#FCE4EC'}
    pct = {s: int(round(SPLIT_RATIO[s] * 100)) for s in SPLITS}
    for x, s in zip(xs, SPLITS):
        c = SPLIT_COUNTS[s]
        box(x, 4.3, w, 1.1, f"{s.upper()} ({pct[s]}%)\n{c['images']} citra · {c['patients']} pasien\n"
            f"Card {c['images_cardiomegaly']} · Non {c['images_non_cardiomegaly']}", colors[s], 9.5, True)
        arrow(7.5, 6.0, x + w / 2, 5.4)

    roles = {
        'train': 'Melatih 3 backbone\n(EffNetB4 · DenseNet121 · ResNet50V2)\nFocal loss · 2 fase',
        'val'  : 'Checkpoint & early stopping\n(max val-acc Fase 1+2)\nBobot FPPR — Pers. (1)',
        'meta' : 'Prediksi backbone (unseen) →\nlatih XGBoost (CV per pasien)\nPilih model final + threshold',
        'test' : 'Dipakai SATU KALI\nEvaluasi akhir 14 konfigurasi\nCI bootstrap · DeLong · McNemar',
    }
    for x, s in zip(xs, SPLITS):
        box(x, 2.2, w, 1.5, roles[s], colors[s], 9)
        arrow(x + w / 2, 4.3, x + w / 2, 3.7)

    # aliran model
    arrow(xs[0] + w, 2.95, xs[1], 2.95, color='#1565C0')
    arrow(xs[1] + w, 2.95, xs[2], 2.95, color='#1565C0')
    arrow(xs[2] + w, 2.95, xs[3], 2.95, color='#C62828')
    ax.text(xs[3] - 0.2, 3.85, 'model & threshold\nterkunci', ha='center', fontsize=8,
            color='#C62828', fontweight='bold')
    if 'EXT_INFO' in globals():
        nA = EXT_INFO['A']['n_images']; nB = EXT_INFO['B']['n_images']
        box(7.6, 0.25, 7.1, 1.45,
            f'EKSTERNAL — CheXpert (Stanford), tanpa retraining\n'
            f'A: Cardiomegaly tunggal vs No Finding ({nA} citra)\n'
            f'B: Cardiomegaly (termasuk komorbid) vs semua non-kardiomegali ({nB} citra)\n'
            f'Model, bobot & threshold terkunci dari partisi meta', '#EDE7F6', 9)
        arrow(xs[3] + w / 2, 2.2, 11.15, 1.7, color='#6A1B9A')
    if 'CTR_INFO' in globals():
        box(0.3, 0.25, 7.0, 1.45,
            'CTR otomatis = MHD / MTD (segmentasi jantung & paru, QC)\n'
            'Baseline klinis CTR > 0.50 pada split yang sama\n'
            'Analisis galat: distribusi CTR pada TP / TN / FP / FN', '#E0F2F1', 9)
        arrow(xs[3] + w / 2 - 0.4, 2.2, 3.8, 1.7, color='#00695C')
    box(0.3, -1.75, 14.4, 1.2,
        'Tidak ada keputusan (checkpoint, bobot, hyperparameter, pemilihan model, threshold) yang memakai partisi TEST.\n'
        'Meta-learner hanya melihat prediksi backbone pada citra yang tidak dipakai untuk melatih maupun memilih checkpoint backbone.',
        '#F5F5F5', 9.5)
    ax.set_title('CELM-Cardio — Protokol Pengembangan & Evaluasi (patient-disjoint)', fontsize=13, fontweight='bold')
    plt.savefig(path_png, dpi=300, bbox_inches='tight'); plt.savefig(path_pdf, bbox_inches='tight')
    plt.show()

draw_protocol(os.path.join(OUTPUT, 'figures', 'protocol_diagram.png'),
              os.path.join(OUTPUT, 'figures', 'protocol_diagram.pdf'))

---
## 💾 SEL 27 — Artefak Reprodusibilitas (R1-11)

Disimpan di folder `results/` dan `models/`:
`split_manifest.csv` · `predictions_{val,meta,test}.csv` (per sampel) · `final_metrics.csv` · `statistical_tests.csv` · `external_chexpert_metrics.csv` · `predictions_chexpert_{A,B}.csv` · `ctr_measurements.csv` · `ctr_analysis.json` · `val_acc_fppr.json` · `experiment_config.json` · `environment.txt` · checkpoint backbone `.keras` · model XGBoost `.pkl` · semua figure.

In [ ]:
# ── Prediksi per sampel ───────────────────────────────────────────────────────
for s in EVAL_SPLITS:
    d = pd.DataFrame({'file': FILES[s],
                      'image_index': [os.path.basename(f) for f in FILES[s]],
                      'patient_id': PID[s], 'split': s, 'y_true': Y[s]})
    for _, k, _, _ in BACKBONES:
        d[f'p_{k}'] = P[k][s]['plain'][:, 1]
        if 'tta' in P[k][s]:
            d[f'p_{k}_tta'] = P[k][s]['tta'][:, 1]
    if s in PROB:
        for c in CONFIGS:
            col = 'p_' + c.replace(' +TTA', '_tta').replace('(', '_').replace(')', '').replace('+', '_')
            if s == 'meta' and c.startswith('XGB'):
                col += '_oof'          # prediksi nested-CV out-of-fold
            d[col] = PROB[s][c]
    d.to_csv(os.path.join(OUTPUT, 'results', f'predictions_{s}.csv'), index=False)
print('✅ predictions_{val,meta,test}.csv tersimpan (kolom XGB di meta = out-of-fold)')

# ── Environment ───────────────────────────────────────────────────────────────
env_lines = [f'python      : {sys.version}', f'platform    : {platform.platform()}',
             f'tensorflow  : {tf.__version__}', f'keras       : {keras.__version__}',
             f'xgboost     : {xgb.__version__}', f'numpy       : {np.__version__}',
             f'pandas      : {pd.__version__}', f'shap        : {shap.__version__}',
             f'opencv      : {cv2.__version__}',
             f'GPU         : {[d.name for d in tf.config.list_physical_devices("GPU")]}']
try:
    import sklearn; env_lines.insert(5, f'scikit-learn: {sklearn.__version__}')
    freeze = subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout
except Exception as ex:
    freeze = f'pip freeze gagal: {ex}'
with open(os.path.join(OUTPUT, 'results', 'environment.txt'), 'w') as fh:
    fh.write('\n'.join(env_lines) + '\n\n# pip freeze\n' + freeze)
print('\n'.join(env_lines))

# ── Konfigurasi eksperimen lengkap ────────────────────────────────────────────
cfg = {
    'version': 'v8-revisi-reviewer',
    'dataset': 'NIH ChestX-ray14 (khanfashee/nih-chest-x-ray-14-224x224-resized)',
    'classes': {'positive': 'Cardiomegaly (single label)', 'negative': 'No Finding'},
    'seed': SEED, 'n_per_class_cap': N_PER_CLASS,
    'split': {'level': 'patient', 'stratified_on': 'patient has any Cardiomegaly image',
              'ratio': SPLIT_RATIO, 'counts': SPLIT_COUNTS,
              'roles': {'train': 'fit backbones', 'val': 'checkpoint/early stopping + FPPR weights',
                        'meta': 'fit XGBoost + model selection + operating threshold',
                        'test': 'single final evaluation'}},
    'training': {'img_size': IMG_SIZE, 'batch': BATCH, 'lr_head': LR_HEAD, 'lr_fine': LR_FINE,
                 'epochs_head': EPOCHS_HEAD, 'epochs_fine': EPOCHS_FINE,
                 'loss': 'focal (gamma=2.0, alpha=0.25)', 'class_weight': CLS_WT,
                 'aug_mode': AUG_MODE,
                 'early_stopping': {'phase1_patience': 5, 'phase2_patience': 12, 'monitor': 'val_accuracy'},
                 'checkpoint_rule': 'max val_accuracy over phase 1 + phase 2; phase-2 '
                                    'ModelCheckpoint initial_value_threshold = phase-1 best'},
    'checkpoints': CKPT, 'val_accuracy': VAL_ACC,
    'fppr': {'equation': 'w_k = Acc_val_k / sum_j Acc_val_j', 'weights': W},
    'tta': {'n': TTA_N, 'seed': TTA_SEED, 'transforms': TTA_TRANSFORMS},
    'xgboost': {'grid': XGB_GRID, 'cv': f'StratifiedGroupKFold({CV_FOLDS}) by patient on meta partition',
                'oof_for_selection': 'nested CV', 'models': XGB_INFO},
    'model_selection': {'criterion': 'max AUC on meta partition (4 d.p.), tie-break: min Brier on meta',
                        'meta_auc': META_AUC, 'meta_brier': META_BRIER,
                        'final_model': FINAL_MODEL, 'threshold_rule': 'Youden J on meta partition',
                        'thresholds': THR},
    'evaluation': {'test_used_once': True, 'bootstrap': f'{N_BOOT} patient-level resamples',
                   'tests': 'DeLong (AUC), exact McNemar (accuracy @0.5), Holm correction',
                   'calibration': f'Brier, ECE ({ECE_BINS} equal-width bins)'},
    'test_results_final': {m: EVAL[FINAL_MODEL]['point'][m] for m in METRICS},
    'external_chexpert': globals().get('EXT_INFO'),
    'external_results_final': ({s: {m: EXT_EVAL[s][FINAL_MODEL]['point'][m] for m in METRICS}
                                for s in EXT_EVAL} if 'EXT_EVAL' in globals() else None),
    'ctr': globals().get('CTR_INFO'),
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}
json.dump(cfg, open(os.path.join(OUTPUT, 'experiment_config.json'), 'w'), indent=2, default=str)
print(f'\n✅ experiment_config.json tersimpan di {OUTPUT}')
print('📁 Isi results/:', sorted(os.listdir(os.path.join(OUTPUT, 'results'))))

---
## 🔮 SEL 28 — Inferensi Satu Citra (Model Final) & Ringkasan

In [ ]:
def predict_xray(img_path: str, cfg: str = None) -> dict:
    """Prediksi satu citra memakai konfigurasi final (default) atau konfigurasi lain."""
    cfg = cfg or FINAL_MODEL
    use_tta = cfg.endswith(' +TTA'); base = cfg.replace(' +TTA', '')
    img_rgb = cv2.cvtColor(cv2.resize(cv2.imread(img_path), IMG_SIZE), cv2.COLOR_BGR2RGB)
    img_f = img_rgb.astype(np.float32)

    def pred_one(img):
        return [MODELS[k].predict(pre(img.copy())[None], verbose=0)
                for _, k, _, pre in BACKBONES]

    if use_tta:
        aug = ImageDataGenerator(**TTA_TRANSFORMS)
        preds = [pred_one(aug.random_transform(img_f, seed=TTA_SEED + i)) for i in range(TTA_N)]
        Pe, Pd, Pr = [np.mean([p[j] for p in preds], axis=0) for j in range(3)]
    else:
        Pe, Pd, Pr = pred_one(img_f)

    if base.startswith('XGB'):
        X = meta_features(Pe, Pd, Pr, with_fppr=(base == 'XGB(BB+FPPR)'))
        p = float(XGB_MODELS[cfg].predict_proba(X)[0, 1])
    else:
        p = float(simple_ensembles(Pe, Pd, Pr)[base][0])
    thr = THR[cfg]
    res = {'config': cfg, 'p_cardiomegaly': round(p, 4),
           'pred@0.5': CLASSES[int(p >= 0.5)], f'pred@{thr:.3f}': CLASSES[int(p >= thr)],
           'catatan': 'Prototipe riset — bukan alat diagnosis klinis.'}
    print(res)
    return res

print('✅ predict_xray() siap — contoh: predict_xray("path/foto.png")')

print('\n' + '═' * 78)
print(f'🏆 RINGKASAN — CELM-Cardio v7 (AUG_MODE={AUG_MODE})')
print('═' * 78)
print(f'Model final (dipilih di partisi meta): {FINAL_MODEL}')
for c in CONFIGS:
    e = EVAL[c]['point']; lo, hi = EVAL[c]['ci']['AUC']
    print(f" {'★' if c == FINAL_MODEL else ' '} {c:22s} AUC={e['AUC']:.4f} [{lo:.4f}–{hi:.4f}]  "
          f"Acc={e['Accuracy']*100:.2f}%  Sens={e['Sensitivity']*100:.1f}%  Spec={e['Specificity']*100:.1f}%")
print('═' * 78)

---
## 🧪 SEL 29 — Perbandingan Skenario Augmentasi (Ablation)

Jalankan notebook 3× dengan `AUG_MODE = 'none'`, `'flip'`, `'noflip'` (SEL 6). Split per pasien identik (seed sama), sehingga hasil bisa dibandingkan langsung. **Catatan:** memilih skenario augmentasi terbaik sebaiknya memakai AUC **meta** (`AUC_meta`), bukan AUC test.

In [ ]:
abl = []
for mode in ['none', 'flip', 'noflip']:
    p = os.path.join(OUTPUT_BASE, f'aug_{mode}', 'results', 'final_metrics.csv')
    if not os.path.exists(p):
        print(f'⏳ Skenario {mode!r} belum dijalankan'); continue
    m = pd.read_csv(p); m.insert(0, 'AUG_MODE', mode)
    abl.append(m)

if abl:
    df_abl = pd.concat(abl, ignore_index=True)
    df_abl.to_csv(os.path.join(OUTPUT_BASE, 'ablation_augmentasi.csv'), index=False)
    cols = [c for c in ['none', 'flip', 'noflip'] if c in df_abl['AUG_MODE'].unique()]
    for metric in ['AUC_meta', 'AUC', 'Accuracy']:
        piv = df_abl.pivot_table(index='Model', columns='AUG_MODE', values=metric, aggfunc='first')
        print(f'\n🎯 {metric} per model × skenario:')
        print(piv.reindex(index=CONFIGS, columns=cols).round(4).to_string())
    print(f'\n💾 {os.path.join(OUTPUT_BASE, "ablation_augmentasi.csv")}')